# Quadriga — Rice Leaf Blast Detection (VGG16) Notebook

It is organised into the modules in the outline: **Pre-processing**, **EDA**, **Dataset Splitting**, **Training (VGG16 baseline)**, **Feature Extraction for PLSR/XGBoost**, **Validation**, **Testing**, and **Reporting**.

## Verify GPU Usage

In [ ]:
import tensorflow as tf

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus}")
    # Test GPU operation
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
        c = tf.matmul(a, b)
        print("GPU matrix multiplication test passed!")
        print(c.numpy())
else:
    print("No GPU detected")

## Library and Initializaations

In [ ]:
# Imports and Configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageFilter
import hashlib
import shutil
import time
import json
import joblib
import cv2
import shap

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model

# Import KerasTuner
import keras_tuner as kt

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# XGBoost (if available)
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. PLSR will be used as alternative.")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set paths to datasets
DATA_DIR_FOREIGN = "datasets/RiceLeaf/GlobalRiceLeaf/shayanriyaz"
DATA_DIR_LOCAL = "datasets/RiceLeaf/LocalRiceLeaf/ZAMBALI_RICE_DATASET_V3"
OUTPUT_BASE = "output/Output_Base"

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Configuration
IMG_EXTS = ('.jpg', '.jpeg', '.png')
TARGET_SIZE = (224, 224)  # VGG16 input size
BATCH_SIZE = 16
EPOCHS = 50

print("Imports and configuration completed successfully.")

### Without Augmentation Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - WITHOUT DATA AUGMENTATION
# =============================================================================

# Set output directory for non-augmented results
OUTPUT_BASE = "output/Output_Base/without_augmentation"

# Set augmentation flag to False
run_aug = False

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("="*60)
print("CONFIGURATION: WITHOUT DATA AUGMENTATION")
print("="*60)
print(f"Output directory: {OUTPUT_BASE}")
print(f"Data augmentation enabled: {run_aug}")
print("All results will be saved without augmented data")
print("="*60)

### With Augmentation Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - WITH DATA AUGMENTATION
# =============================================================================

# Set output directory for augmented results
OUTPUT_BASE = "output/Output_Base/with_augmentation"

# Set augmentation flag to True
run_aug = True

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("="*60)
print("CONFIGURATION: WITH DATA AUGMENTATION")
print("="*60)
print(f"Output directory: {OUTPUT_BASE}")
print(f"Data augmentation enabled: {run_aug}")
print("Data augmentation will be applied to training set")
print("="*60)

## 1) Pre-processing Module

Steps:

1. Extract RGB images (PNG/JPG) into numpy arrays.
2. Remove duplicates and blurred images (automatic).
3. Annotate/Labeling of Images
4. Generate CSV or JSON with metadata and labels.
5. Perform EDA
6. Data Augmentation for Class Balance

### 1.1 Image Extraction and Processing

In [ ]:
def load_and_preprocess_image(image_path, target_size=TARGET_SIZE):
    """Load and preprocess image for CNN"""
    try:
        image = Image.open(image_path)
        image = image.resize(target_size)
        image_array = np.array(image)

        if len(image_array.shape) == 2:
            image_array = np.stack([image_array] * 3, axis=-1)
        elif image_array.shape[2] == 4:
            image_array = image_array[:, :, :3]

        return image_array.astype(np.float32) / 255.0  # Normalize to [0,1]
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None

def extract_all_images(data_dir):
    """Extract all images from directory"""
    image_files = []

    for ext in IMG_EXTS:
        files = Path(data_dir).rglob(f"*{ext}")
        image_files.extend(files)

    unique_files = list(set(str(img_path) for img_path in image_files))

    print(f"Found {len(unique_files)} images in {data_dir}")
    return unique_files

# Extract images from both datasets with the function
foreign_images = extract_all_images(DATA_DIR_FOREIGN)
ph_images = extract_all_images(DATA_DIR_LOCAL)

print("Image extraction completed.")

### 1.2 Duplicate and Blur Detection

In [ ]:
def calculate_image_hash(image_path):
    #Calculate MD5 hash of an image file to identify duplicates
    try:
        with open(image_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception as e:
        print(f"Error reading {image_path}: {e}")
        return None

def detect_blur_image_pil(image_path, threshold=100):
    # Detect blur using PIL - variance of Laplacian approximation
    try:
        image = Image.open(image_path).convert('L')
        image_array = np.array(image)

        laplacian_var = np.var(image_array)

        if np.random.random() < 0.01:
            print(f"Debug - {Path(image_path).name}: Laplacian variance = {laplacian_var:.2f}")

        return laplacian_var < threshold
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return True

def filter_images(image_paths, dataset_name):
    """Filter images for duplicates and blur"""
    print(f"Filtering {dataset_name} dataset...")

    image_hashes = {}
    valid_images = []
    invalid_images = []

    for img_path in image_paths:
        img_hash = calculate_image_hash(img_path)
        if img_hash is None:
            invalid_images.append(img_path)
            continue

        if img_hash in image_hashes:
            invalid_images.append(img_path)
            continue

        if detect_blur_image_pil(img_path):
            invalid_images.append(img_path)
            continue

        image_hashes[img_hash] = img_path
        valid_images.append(img_path)

    print(f"Valid images: {len(valid_images)}, Invalid images: {len(invalid_images)}")
    return valid_images, invalid_images

# Filter both datasets
foreign_valid, foreign_invalid = filter_images(foreign_images, "foreign")
ph_valid, ph_invalid = filter_images(ph_images, "Philippines")

print("Duplicate and blur detection completed.")

### 1.3 Image Segmentation

#### Background Removal

#### CLAHE

In [ ]:
#################
# 2025 Deepseek #
#################
def apply_clahe_enhancement(image_array):
    """Apply CLAHE enhancement to RGB image without grayscale conversion"""
    try:
        # Convert to 8-bit for OpenCV processing
        img_uint8 = (image_array * 255).astype(np.uint8)

        # Convert to LAB color space to apply CLAHE on luminance channel
        lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        # Apply CLAHE on the luminance channel
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_clahe = clahe.apply(l)

        # Merge back the enhanced luminance with original a and b channels
        lab_clahe = cv2.merge([l_clahe, a, b])

        # Convert back to RGB
        enhanced_rgb = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)

        # Convert back to float and normalize
        result_image = enhanced_rgb.astype(np.float32) / 255.0

        return result_image

    except Exception as e:
        print(f"Error in CLAHE enhancement: {e}")
        return image_array

def apply_clahe_to_dataset(valid_images, dataset_name):
    """Apply CLAHE enhancement to all valid images"""
    print(f"Applying CLAHE enhancement to {dataset_name} dataset...")

    processed_images = []

    for i, img_path in enumerate(valid_images):
        if i % 500 == 0:
            print(f"  Processed {i}/{len(valid_images)} images...")

        try:
            # Load original image
            original_img = load_and_preprocess_image(img_path)
            if original_img is None:
                continue

            # Apply CLAHE enhancement
            enhanced_img = apply_clahe_enhancement(original_img)

            processed_images.append({
                'original_path': img_path,
                'processed_image': enhanced_img,  # 3-channel CLAHE enhanced RGB
                'is_enhanced': True
            })

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            # Fallback to original image
            original_img = load_and_preprocess_image(img_path)
            if original_img is not None:
                processed_images.append({
                    'original_path': img_path,
                    'processed_image': original_img,
                    'is_enhanced': False
                })

    print(f"✓ CLAHE enhancement completed for {dataset_name}: {len(processed_images)} images")
    return processed_images

def visualize_clahe_samples(processed_images, dataset_name, num_samples=5):
    """Visualize samples of CLAHE enhancement results"""
    print(f"Visualizing CLAHE enhancement for {dataset_name}...")

    samples = min(num_samples, len(processed_images))
    fig, axes = plt.subplots(samples, 2, figsize=(10, 5 * samples))

    if samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(samples):
        img_data = processed_images[i]

        # Load original for comparison
        original_img = load_and_preprocess_image(img_data['original_path'])

        # Original image
        axes[i, 0].imshow(original_img)
        axes[i, 0].set_title('Original Image')
        axes[i, 0].axis('off')

        # CLAHE enhanced RGB
        axes[i, 1].imshow(img_data['processed_image'])
        axes[i, 1].set_title('CLAHE Enhanced')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, f'clahe_enhancement_{dataset_name}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

#### Disable Preprocessing Image

In [ ]:
##########################################################################

# Apply background removal to both datasets
#print("=== BACKGROUND REMOVAL AND THRESHOLDING ===")
#foreign_processed = apply_background_removal_to_dataset(foreign_valid, "foreign")
#ph_processed = apply_background_removal_to_dataset(ph_valid, "Philippines")

# Visualize results
#visualize_background_removal_samples(foreign_processed, "foreign")
#visualize_background_removal_samples(ph_processed, "Philippines")

#print("Background removal and thresholding completed.")

##########################################################################

##########################################################################

print("=== CLAHE ENHANCEMENT ===")
foreign_processed = apply_clahe_to_dataset(foreign_valid, "foreign")
ph_processed = apply_clahe_to_dataset(ph_valid, "Philippines")

# Visualize results
visualize_clahe_samples(foreign_processed, "foreign")
visualize_clahe_samples(ph_processed, "Philippines")

print("CLAHE enhancement completed.")

##########################################################################

##########################################################################

# Disabled Image Processing
# Create dummy processed images (just original images)
#foreign_processed = create_dummy_processed_images(foreign_valid)
#ph_processed = create_dummy_processed_images(ph_valid)

#print("Background removal step skipped - using original images")

##########################################################################

### 1.4 Annotate/Labeling of Images & 1.4 Generate CSV/JSON

In [ ]:
def create_annotation_format(filename, label, origin, unique_id, transformation=None):
    base_name = f"{label}_{origin}_{unique_id:04d}"
    if transformation:
        base_name += f"_{transformation}"
    return f"{base_name}{Path(filename).suffix}"

def auto_detect_label(image_path):
    path_lower = image_path.lower()

    if 'blast' in path_lower or 'disease' in path_lower or 'infected' in path_lower:
        return 'LEAFBLAST'
    elif 'healthy' in path_lower or 'normal' in path_lower:
        return 'HEALTHY'
    else:
        return 'UNKNOWN'

def create_metadata(valid_images, invalid_images, dataset_name):
    metadata = []
    unique_id = 1

    for img_data in processed_images:
        img_path = img_data['original_path']
        label = auto_detect_label(img_path)
        origin = 'foreignI' if dataset_name == 'foreign' else 'LOCAL'

        annotated_name = create_annotation_format(
            Path(img_path).name,
            label,
            origin,
            unique_id
        )

        metadata.append({
            'original_path': img_path,
            'annotated_name': annotated_name,
            'label': label,
            'origin': origin,
            'unique_id': unique_id,
            'dataset': dataset_name,
            'is_background_removed': False,
            'has_background_mask': False,
            'is_enhanced': img_data.get('is_enhanced', True)  # Track CLAHE status
        })
        unique_id += 1

    # Save to CSV
    df = pd.DataFrame(metadata)
    csv_path = os.path.join(OUTPUT_BASE, f'{dataset_name}_metadata_processed.csv')
    df.to_csv(csv_path, index=False)

    # Save to JSON
    json_path = os.path.join(OUTPUT_BASE, f'{dataset_name}_metadata_processed.json')
    with open(json_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved {len(metadata)} PROCESSED records for {dataset_name} dataset")
    print(f"All images are CLAHE-enhanced: {all([item.get('is_enhanced', True) for item in metadata])}")

    return df

foreign_metadata = create_metadata(foreign_processed, foreign_invalid, 'foreign')
ph_metadata = create_metadata(ph_processed, ph_invalid, 'philippines')

print("Image annotation and metadata generation completed.")

### 1.5 Exploratory Data Analysis (EDA)

In [ ]:
#################
# 2025 Deepseek #
#################
def perform_eda(foreign_metadata, ph_metadata):
    # Combine both datasets (these are now DataFrames, not lists)
    combined_meta = pd.concat([foreign_metadata, ph_metadata], ignore_index=True)

    # Plot combined distribution
    plt.figure(figsize=(15, 6))

    # Plot 1: Combined origin distribution (pie chart)
    plt.subplot(1, 3, 1)
    origin_counts = combined_meta['origin'].value_counts()
    plt.pie(origin_counts.values, labels=origin_counts.index, autopct='%1.1f%%',
            colors=['lightblue', 'lightcoral'])
    plt.title('Combined Image Distribution by Origin')

    # Plot 2: Combined label distribution (bar chart)
    plt.subplot(1, 3, 2)
    label_counts = combined_meta['label'].value_counts()
    bars = plt.bar(range(len(label_counts)), label_counts.values,
                   color=['red', 'green', 'gray'])
    plt.xticks(range(len(label_counts)), label_counts.index, rotation=45, ha='right')
    plt.title('Combined Image Distribution by Label')

    for i, bar in enumerate(bars):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom')

    # Plot 3: Background removal success
    plt.subplot(1, 3, 3)
    if 'is_background_removed' in combined_meta.columns:
        bg_removal_counts = combined_meta['is_background_removed'].value_counts()

        true_count = bg_removal_counts.get(True, 0)
        false_count = bg_removal_counts.get(False, 0)

        values = [true_count, false_count]
        labels = ['Background Removed', 'Original']
        colors = ['lightgreen', 'lightyellow']

        if sum(values) > 0:
            plt.pie(values, labels=labels, autopct='%1.1f%%', colors=colors)
        else:
            plt.text(0.5, 0.5, 'No Background\nRemoval Data',
                    ha='center', va='center', transform=plt.gca().transAxes)

        plt.title('Background Removal Success Rate')
    else:
        plt.text(0.5, 0.5, 'Background Removal\nData Not Available',
                ha='center', va='center', transform=plt.gca().transAxes)
        plt.title('Background Removal Status')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'combined_data_distribution.png'), dpi=300, bbox_inches='tight')
    plt.show()

    # Print detailed statistics
    print("\n" + "COMBINED DATASET STATISTICS")

    print(f"\nTotal images: {len(combined_meta)}")
    print(f"foreigni images: {len(foreign_metadata)}")
    print(f"Philippines images: {len(ph_metadata)}")

    print("\nOverall class distribution:")
    print(combined_meta['label'].value_counts())

    print("\nforeigni dataset class distribution:")
    print(foreign_metadata['label'].value_counts())

    print("\nPhilippines dataset class distribution:")
    print(ph_metadata['label'].value_counts())

    # Background removal statistics
    if 'is_background_removed' in combined_meta.columns:
        bg_removed_total = combined_meta['is_background_removed'].sum()
        bg_removed_percentage = bg_removed_total / len(combined_meta) * 100
        print(f"\nBackground Removal Statistics:")
        print(f"Images with background removed: {bg_removed_total}/{len(combined_meta)} ({bg_removed_percentage:.1f}%)")

    # Calculate percentages
    print("\nPercentage distribution - Combined:")
    total = len(combined_meta)
    for label, count in combined_meta['label'].value_counts().items():
        print(f"  {label}: {count} ({count/total*100:.1f}%)")

    print("\nPercentage distribution - foreign:")
    total_bd = len(foreign_metadata)
    for label, count in foreign_metadata['label'].value_counts().items():
        print(f"  {label}: {count} ({count/total_bd*100:.1f}%)")

    print("\nPercentage distribution - Philippines:")
    total_ph = len(ph_metadata)
    for label, count in ph_metadata['label'].value_counts().items():
        print(f"  {label}: {count} ({count/total_ph*100:.1f}%)")

    # Return the combined statistics
    return {
        'combined_meta': combined_meta,
        'foreign_counts': foreign_metadata['label'].value_counts(),
        'ph_counts': ph_metadata['label'].value_counts(),
        'combined_counts': combined_meta['label'].value_counts()
    }

# Perform COMBINED EDA for both datasets
combined_stats = perform_eda(foreign_metadata, ph_metadata)

print("Combined Exploratory Data Analysis completed.")

### 1.6 Class Balance Auto

In [ ]:
#################
# 2025 Deepseek #
#################
def visualize_class_balance_comprehensive(foreign_metadata, ph_metadata, foreign_balanced, ph_balanced):
    """Comprehensive visualization of class balance before and after balancing"""
    print("Generating comprehensive class balance visualization...")

    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Before balancing - foreign
    bd_before_counts = foreign_metadata['label'].value_counts()
    axes[0, 0].pie(bd_before_counts.values, labels=bd_before_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[0, 0].set_title('foreign - Before Balancing', fontsize=14, fontweight='bold')

    # After balancing - foreign
    bd_after_counts = foreign_balanced['label'].value_counts()
    axes[0, 1].pie(bd_after_counts.values, labels=bd_after_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[0, 1].set_title('foreign - After Balancing', fontsize=14, fontweight='bold')

    # Before balancing - Philippines
    ph_before_counts = ph_metadata['label'].value_counts()
    axes[0, 2].pie(ph_before_counts.values, labels=ph_before_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[0, 2].set_title('Philippines - Before Balancing', fontsize=14, fontweight='bold')

    # After balancing - Philippines
    ph_after_counts = ph_balanced['label'].value_counts()
    axes[1, 0].pie(ph_after_counts.values, labels=ph_after_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[1, 0].set_title('Philippines - After Balancing', fontsize=14, fontweight='bold')

    # Bar chart comparison - foreign
    x = np.arange(len(bd_before_counts))
    width = 0.35
    axes[1, 1].bar(x - width/2, bd_before_counts.values, width, label='Before', alpha=0.7, color='red')
    axes[1, 1].bar(x + width/2, bd_after_counts.values, width, label='After', alpha=0.7, color='blue')
    axes[1, 1].set_xlabel('Classes')
    axes[1, 1].set_ylabel('Number of Images')
    axes[1, 1].set_title('foreign - Balance Comparison', fontsize=14, fontweight='bold')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(bd_before_counts.index)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Bar chart comparison - Philippines
    x = np.arange(len(ph_before_counts))
    axes[1, 2].bar(x - width/2, ph_before_counts.values, width, label='Before', alpha=0.7, color='red')
    axes[1, 2].bar(x + width/2, ph_after_counts.values, width, label='After', alpha=0.7, color='blue')
    axes[1, 2].set_xlabel('Classes')
    axes[1, 2].set_ylabel('Number of Images')
    axes[1, 2].set_title('Philippines - Balance Comparison', fontsize=14, fontweight='bold')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels(ph_before_counts.index)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comprehensive_class_balance.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Print detailed statistics
    print("\n" + "="*60)
    print("CLASS BALANCE STATISTICS")
    print("="*60)

    # foreign stats
    bd_before_total = len(foreign_metadata)
    bd_after_total = len(foreign_balanced)
    bd_removed = bd_before_total - bd_after_total

    print(f"\nforeign Dataset:")
    print(f"  Before balancing: {bd_before_total} images")
    print(f"  After balancing:  {bd_after_total} images")
    print(f"  Removed samples:  {bd_removed} images")
    print(f"  Balance ratio: {bd_after_counts.min()}/{bd_after_counts.max()} "
          f"({bd_after_counts.min()/bd_after_counts.max()*100:.1f}%)")

    # Philippines stats
    ph_before_total = len(ph_metadata)
    ph_after_total = len(ph_balanced)
    ph_removed = ph_before_total - ph_after_total

    print(f"\nPhilippines Dataset:")
    print(f"  Before balancing: {ph_before_total} images")
    print(f"  After balancing:  {ph_after_total} images")
    print(f"  Removed samples:  {ph_removed} images")
    print(f"  Balance ratio: {ph_after_counts.min()}/{ph_after_counts.max()} "
          f"({ph_after_counts.min()/ph_after_counts.max()*100:.1f}%)")

    # Overall impact
    total_before = bd_before_total + ph_before_total
    total_after = bd_after_total + ph_after_total
    total_removed = total_before - total_after

    print(f"\nOverall Impact:")
    print(f"  Total before balancing: {total_before} images")
    print(f"  Total after balancing:  {total_after} images")
    print(f"  Total removed:          {total_removed} images")
    print(f"  Reduction: {total_removed/total_before*100:.1f}%")

In [ ]:
#################
# 2025 Deepseek #
#################
def balance_classes_undersample(metadata_df, dataset_name):
    """Balance classes by REDUCING majority class to match minority class"""
    print(f"Balancing classes for {dataset_name} dataset (undersampling)...")

    # Get class distribution
    label_counts = metadata_df['label'].value_counts()
    print(f"Current class distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} images")

    # Find the minority class and its count
    minority_label = label_counts.index[-1]  # Last one is the smallest
    minority_count = label_counts.iloc[-1]

    print(f"Minority class: {minority_label} with {minority_count} images")
    print(f"Target count for all classes: {minority_count} images")

    balanced_dfs = []

    for label in label_counts.index:
        class_df = metadata_df[metadata_df['label'] == label]
        current_count = len(class_df)

        if current_count > minority_count:
            # Need to undersample this class (reduce to minority count)
            print(f"Undersampling {label}: {current_count} -> {minority_count} (removing {current_count - minority_count})")

            # Randomly sample without replacement to reduce to minority count
            undersampled = class_df.sample(n=minority_count, replace=False, random_state=42)
            balanced_dfs.append(undersampled)

        elif current_count < minority_count:
            # This shouldn't happen if minority_count is truly the minimum
            print(f"Warning: {label} has {current_count} which is less than minority count {minority_count}")
            balanced_dfs.append(class_df)
        else:
            # Already at the target count
            print(f"Keeping {label}: {current_count} images (already balanced)")
            balanced_dfs.append(class_df)

    # Combine all balanced classes
    balanced_df = pd.concat(balanced_dfs, ignore_index=True)

    # Verify new distribution
    balanced_counts = balanced_df['label'].value_counts()
    print(f"Balanced class distribution:")
    for label, count in balanced_counts.items():
        print(f"  {label}: {count} images")

    # Calculate balance statistics
    original_total = len(metadata_df)
    balanced_total = len(balanced_df)
    removed_samples = original_total - balanced_total

    print(f"Balance summary:")
    print(f"  Original total: {original_total}")
    print(f"  Balanced total: {balanced_total}")
    print(f"  Removed samples: {removed_samples}")
    print(f"  Balance ratio: {balanced_counts.min()}/{balanced_counts.max()} "
          f"({balanced_counts.min()/balanced_counts.max()*100:.1f}%)")

    return balanced_df

# Balance both datasets using undersampling
print("\nBalancing foreign dataset...")
foreign_balanced = balance_classes_undersample(foreign_metadata, "foreign")

print("\nBalancing Philippines dataset...")
ph_balanced = balance_classes_undersample(ph_metadata, "Philippines")

# FIRST: Visualize comparison (original vs balanced)
print("Generating enhanced class balance visualization...")
visualize_class_balance_comprehensive(foreign_metadata, ph_metadata, foreign_balanced, ph_balanced)

# THEN: Update the metadata with balanced versions
foreign_metadata = foreign_balanced
ph_metadata = ph_balanced

print("\nClass balancing completed!")
print("="*60)

### 1.7 Dataset Splitting

In [ ]:
def combine_and_stratified_split(foreign_metadata, ph_metadata):
    """Combine datasets immediately and perform stratified train/test split"""
    print("="*60)
    print("COMBINE DATASETS + STRATIFIED SPLIT")
    print("="*60)
    
    # Combine both datasets
    combined_df = pd.concat([foreign_metadata, ph_metadata], ignore_index=True)
    print(f"Total combined images: {len(combined_df)}")
    print(f"Class distribution in combined data:")
    print(combined_df['label'].value_counts())
    
    # Split into train+val and test (80/20) with stratification
    train_val_df, test_df = train_test_split(
        combined_df,
        test_size=0.2,
        random_state=42,
        stratify=combined_df['label']
    )
    
    # Further split train+val into train and validation (75/25 of train_val)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=0.25,  # 0.25 * 0.8 = 0.2 of total for validation
        random_state=42,
        stratify=train_val_df['label']
    )
    
    print(f"\nFinal splits:")
    print(f"Train: {len(train_df)} images ({len(train_df)/len(combined_df)*100:.1f}%)")
    print(f"Validation: {len(val_df)} images ({len(val_df)/len(combined_df)*100:.1f}%)")
    print(f"Test: {len(test_df)} images ({len(test_df)/len(combined_df)*100:.1f}%)")
    
    # Verify stratification maintained class distribution
    print(f"\nClass distribution in Train: {train_df['label'].value_counts()}")
    print(f"Class distribution in Val: {val_df['label'].value_counts()}")
    print(f"Class distribution in Test: {test_df['label'].value_counts()}")
    
    return train_df, val_df, test_df

# Replace your current split_datasets function call with:
print("Applying new stratified split strategy...")
train_df, val_df, test_df = combine_and_stratified_split(foreign_balanced, ph_balanced)

### 1.8 Dataset Augmentation

In [ ]:
#################
# 2025 Deepseek #
#################
if run_aug:
    import random

    print(f"Data augmentation ENABLED")
    print(f"Output folder set to: {OUTPUT_BASE}")

    def visualize_augmentation_samples(original_paths, augmented_data, processed_dict=None, num_samples=3):
        """Visualize original images and their augmented versions using processed images when available"""
        print(f"\nVisualizing augmentation samples...")

        # Get unique original images from augmented data
        unique_original_paths = list(set([item['original_path'] for item in augmented_data]))

        # Select random samples
        if len(unique_original_paths) < num_samples:
            num_samples = len(unique_original_paths)

        selected_paths = random.sample(unique_original_paths, num_samples)

        # Create figure
        fig, axes = plt.subplots(num_samples, 6, figsize=(20, 4 * num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)

        for i, original_path in enumerate(selected_paths):
            # Get original image - use processed if available, otherwise load original
            if processed_dict is not None and original_path in processed_dict:
                original_img = processed_dict[original_path]
                image_source = "Processed"
            else:
                original_img = load_and_preprocess_image(original_path)
                image_source = "Original"

            # Get augmented versions of this image
            aug_versions = [item for item in augmented_data if item['original_path'] == original_path]

            # Display original image
            axes[i, 0].imshow(original_img)
            axes[i, 0].set_title(f'{image_source} Image\n{Path(original_path).name}', fontsize=10)
            axes[i, 0].axis('off')

            # Display up to 5 augmented versions
            for j in range(min(5, len(aug_versions))):
                aug_img = aug_versions[j]['augmented_image']
                aug_type = aug_versions[j]['augmentation_type']

                axes[i, j+1].imshow(aug_img)
                axes[i, j+1].set_title(f'Augmented\n{aug_type}', fontsize=10)
                axes[i, j+1].axis('off')

            # If we have less than 5 augmentations, hide empty subplots
            for j in range(len(aug_versions) + 1, 6):
                axes[i, j].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_BASE, 'augmentation_samples.png'),
                    dpi=300, bbox_inches='tight')
        plt.show()

    def apply_data_augmentation_with_processed(train_df, augmentation_percentage=0.3, processed_dict=None):
        """Apply data augmentation using processed images when available"""
        print(f"\nApplying data augmentation ({augmentation_percentage*100}% increase)...")

        def adjust_brightness(image, factor):
            """Adjust image brightness"""
            return np.clip(image * factor, 0, 1)

        def adjust_contrast(image, factor):
            """Adjust image contrast"""
            mean = np.mean(image, axis=(0,1), keepdims=True)
            return np.clip((image - mean) * factor + mean, 0, 1)

        # Define all augmentation types
        augmentation_types = [
            ('flip_h', lambda img: np.fliplr(img)),
            ('flip_v', lambda img: np.flipud(img)),
            ('rot90', lambda img: np.rot90(img, 1)),
            ('rot180', lambda img: np.rot90(img, 2)),
            ('rot270', lambda img: np.rot90(img, 3)),
            ('bright_high', lambda img: adjust_brightness(img, 1.3)),
            ('bright_low', lambda img: adjust_brightness(img, 0.7)),
            ('contrast_high', lambda img: adjust_contrast(img, 1.5)),
            ('contrast_low', lambda img: adjust_contrast(img, 0.7)),
        ]

        print(f"Available augmentation types: {len(augmentation_types)}")
        print(f"Using processed images for augmentation: {processed_dict is not None}")

        augmented_data = []

        # Process each class separately
        for label in train_df['label'].unique():
            print(f"\nAugmenting {label} class...")
            class_images = train_df[train_df['label'] == label]
            original_count = len(class_images)

            # Calculate number of augmented samples needed
            samples_needed = max(
                len(augmentation_types),  # Minimum: one of each augmentation type
                int(original_count * augmentation_percentage)  # Percentage of original
            )

            print(f"  Original images: {original_count}")
            print(f"  Target augmented samples: {samples_needed}")

            # First pass: Ensure we have at least one of each augmentation type
            base_augmentations = []
            available_images = class_images.copy().reset_index(drop=True)

            for aug_idx, (aug_type_name, aug_func) in enumerate(augmentation_types):
                if aug_idx < len(available_images):
                    # Use a different image for each augmentation type
                    row = available_images.iloc[aug_idx]
                else:
                    # If we have more augmentation types than images, reuse images
                    row = available_images.sample(1).iloc[0]

                try:
                    # Use processed image if available, otherwise load original
                    if processed_dict is not None and row['original_path'] in processed_dict:
                        original_img = processed_dict[row['original_path']]
                    else:
                        original_img = load_and_preprocess_image(row['original_path'])

                    if original_img is None:
                        continue

                    aug_img = aug_func(original_img)

                    base_augmentations.append({
                        'original_path': row['original_path'],
                        'augmented_image': aug_img,
                        'augmentation_type': aug_type_name,
                        'label': row['label'],
                        'origin': row['origin'],
                        'unique_id': row['unique_id'],
                        'row_index': aug_idx
                    })

                except Exception as e:
                    print(f"Error in base augmentation for {row['original_path']}: {e}")

            # Second pass: Fill remaining needed samples
            additional_augmentations = []
            augment_count = len(base_augmentations)

            while augment_count < samples_needed:
                # Randomly select an image and augmentation
                random_row = class_images.sample(1).iloc[0]
                aug_type_name, aug_func = random.choice(augmentation_types)

                try:
                    # Use processed image if available
                    if processed_dict is not None and random_row['original_path'] in processed_dict:
                        original_img = processed_dict[random_row['original_path']]
                    else:
                        original_img = load_and_preprocess_image(random_row['original_path'])

                    if original_img is None:
                        continue

                    aug_img = aug_func(original_img)

                    additional_augmentations.append({
                        'original_path': random_row['original_path'],
                        'augmented_image': aug_img,
                        'augmentation_type': aug_type_name,
                        'label': random_row['label'],
                        'origin': random_row['origin'],
                        'unique_id': random_row['unique_id'],
                        'row_index': len(class_images) + augment_count
                    })

                    augment_count += 1
                    if augment_count % 10 == 0:
                        print(f"  Generated {augment_count}/{samples_needed} augmentations", end='\r')

                except Exception as e:
                    print(f"Error in additional augmentation: {e}")

            # Combine and create final entries
            all_augmentations = base_augmentations + additional_augmentations

            for i, aug_data in enumerate(all_augmentations):
                augmented_name = create_annotation_format(
                    Path(aug_data['original_path']).name,
                    aug_data['label'],
                    aug_data['origin'],
                    aug_data['unique_id'] * 1000 + i,
                    f"AUG_{aug_data['augmentation_type']}"
                )

                augmented_data.append({
                    'original_path': aug_data['original_path'],
                    'annotated_name': augmented_name,
                    'label': aug_data['label'],
                    'origin': aug_data['origin'],
                    'unique_id': aug_data['unique_id'] * 1000 + i,
                    'augmented_image': aug_data['augmented_image'],
                    'is_augmented': True,
                    'augmentation_type': aug_data['augmentation_type']
                })

            print(f"  ✓ Generated {len(all_augmentations)} augmentations for {label}")

        print(f"\nGenerated {len(augmented_data)} augmented images total")

        return augmented_data

    # Create processed image dictionary for augmentation
    foreign_processed_dict = {}
    if 'foreign_processed' in locals() and foreign_processed is not None:
        for item in foreign_processed:
            foreign_processed_dict[item['original_path']] = item['processed_image']
        print(f"Using {len(foreign_processed_dict)} processed images for augmentation")
    else:
        print("No processed images available - using original images for augmentation")

    # Store original training data
    original_train_size = len(train_df)
    original_train_df = train_df.copy()

    # Ensure original_train_df has is_augmented column set to False
    if 'is_augmented' not in original_train_df.columns:
        original_train_df['is_augmented'] = False
    else:
        original_train_df['is_augmented'] = original_train_df['is_augmented'].astype(bool)

    # Apply augmentation WITH processed images
    augmentation_percentage = 0.3  # 30% increase for both classes
    print(f"Using augmentation percentage: {augmentation_percentage*100}%")

    augmented_data = apply_data_augmentation_with_processed(
        train_df,
        augmentation_percentage,
        processed_dict=foreign_processed_dict
    )
    augmented_df = pd.DataFrame(augmented_data)

    # VISUALIZATION: Show augmentation samples using processed images when available
    visualize_augmentation_samples(
        original_train_df['original_path'].tolist(),
        augmented_data,
        processed_dict=foreign_processed_dict,
        num_samples=4
    )

    # Combine original and augmented data
    print(f"\nCombining datasets:")
    print(f"  Original training data: {len(original_train_df)} rows")
    print(f"  Augmented data: {len(augmented_df)} rows")

    # Ensure both DataFrames have compatible columns
    required_columns = ['original_path', 'annotated_name', 'label', 'origin', 'unique_id', 'is_augmented']

    # Make sure original_train_df has all required columns
    for col in required_columns:
        if col not in original_train_df.columns and col != 'is_augmented':
            original_train_df[col] = None

    # Make sure augmented_df has all required columns
    for col in required_columns:
        if col not in augmented_df.columns and col != 'is_augmented':
            augmented_df[col] = None

    # Now combine
    train_df = pd.concat([original_train_df, augmented_df], ignore_index=True)

    # Ensure is_augmented is boolean
    train_df['is_augmented'] = train_df['is_augmented'].astype(bool)

    print(f"  Final combined dataset: {len(train_df)} rows")
    print(f"  - Original samples: {(~train_df['is_augmented']).sum()}")
    print(f"  - Augmented samples: {train_df['is_augmented'].sum()}")

    # Save information about the augmentation
    augmentation_info = {
        'original_training_size': original_train_size,
        'augmented_samples': len(augmented_df),
        'final_training_size': len(train_df),
        'augmentation_percentage': augmentation_percentage,
        'augmentation_subfolder': OUTPUT_BASE,
        'used_processed_images': len(foreign_processed_dict) > 0
    }

    with open(os.path.join(OUTPUT_BASE, 'augmentation_info.json'), 'w') as f:
        json.dump(augmentation_info, f, indent=2)

    augmented_df.to_csv(os.path.join(OUTPUT_BASE, 'augmented_data.csv'), index=False)

    print(f"\nAugmentation Summary:")
    print(f"Original training set: {original_train_size} images")
    print(f"Augmented samples: {len(augmented_df)} images")
    print(f"Final training set: {len(train_df)} images")
    print(f"Augmentation increased dataset by {len(augmented_df)/original_train_size*100:.1f}%")
    print(f"Used processed images: {len(foreign_processed_dict) > 0}")
    print(f"Output folder: {OUTPUT_BASE}")

    print("Dataset augmentation phase completed.")

else:
    print("="*60)
    print("DATA AUGMENTATION SKIPPED")
    print("="*60)
    print("run_aug is set to False - proceeding without data augmentation")
    print(f"Using original training set: {len(train_df)} images")
    print("="*60)

## 2. CNN Training Module

### 2.0 Data Preparation Cell

In [ ]:
#################
# 2025 Deepseek #
#################
def prepare_training_data_orthogonal(foreign_metadata, ph_metadata, train_df=None, val_df=None, test_df=None,
                                   foreign_processed=None, ph_processed=None):
    """ORTHOGONAL VERSION: Handle both augmented data AND processed images independently"""

    # Independent decisions
    using_processed = foreign_processed is not None and ph_processed is not None
    using_augmented = train_df is not None and 'is_augmented' in train_df.columns and any(train_df['is_augmented'])

    print("="*60)
    print("ORTHOGONAL DATA PREPARATION")
    print("="*60)
    print(f"Image Preprocessing: {'CLAHE-Enhanced' if using_processed else 'Original Images'}")
    print(f"Data Augmentation: {'Enabled' if using_augmented else 'Disabled'}")

    # Create lookup dictionaries for processed images
    foreign_lookup = {}
    ph_lookup = {}

    if using_processed:
        for item in foreign_processed:
            foreign_lookup[item['original_path']] = item['processed_image']
        for item in ph_processed:
            ph_lookup[item['original_path']] = item['processed_image']
        print(f"Processed images available: foreign ({len(foreign_lookup)}), Philippines ({len(ph_lookup)})")

    def get_image(original_path, dataset_type="foreign"):
        """Get image - uses processed version if available, otherwise loads original"""
        if using_processed:
            lookup_dict = foreign_lookup if dataset_type == "foreign" else ph_lookup
            if original_path in lookup_dict:
                return lookup_dict[original_path]
        # Fallback to loading original image
        return load_and_preprocess_image(original_path)

    def safe_array_conversion(image_list):
        """Safely convert list of images to numpy array"""
        if not image_list:
            return np.array([])

        first_shape = image_list[0].shape
        all_same_shape = all(img.shape == first_shape for img in image_list)

        if all_same_shape:
            return np.array(image_list)
        else:
            print(f"Warning: Inconsistent image shapes. First: {first_shape}")
            shapes = [img.shape for img in image_list]
            shape_counts = {}
            for shape in shapes:
                shape_counts[shape] = shape_counts.get(shape, 0) + 1

            most_common_shape = max(shape_counts.items(), key=lambda x: x[1])[0]
            compatible_images = [img for img in image_list if img.shape == most_common_shape]
            print(f"Using {len(compatible_images)}/{len(image_list)} images with consistent shape")
            return np.array(compatible_images)

    # FIXED: Handle the case where we have processed images but no train_df (non-augmented)
    X_train = []
    y_train = []
    X_val = []
    y_val = []
    X_test = []
    y_test = []

    # CASE 1: We have train_df (augmented case OR non-augmented with DataFrame)
    if train_df is not None and val_df is not None:
        print(f"\nPreparing training data from DataFrame ({len(train_df)} rows)...")
        augmented_count = 0
        original_count = 0
        processed_count = 0

        for _, row in train_df.iterrows():
            try:
                # Handle augmented images (pre-computed)
                if using_augmented and row.get('is_augmented', False) and 'augmented_image' in row and row['augmented_image'] is not None:
                    aug_img = row['augmented_image']
                    if isinstance(aug_img, np.ndarray) and aug_img.shape == (224, 224, 3):
                        X_train.append(aug_img)
                        y_train.append(1 if row['label'] == 'LEAFBLAST' else 0)
                        augmented_count += 1
                else:
                    # Handle original images (load from path or use processed)
                    if 'original_path' in row and row['original_path'] is not None:
                        img_array = get_image(row['original_path'], "foreign")
                        if img_array is not None:
                            X_train.append(img_array)
                            y_train.append(1 if row['label'] == 'LEAFBLAST' else 0)
                            original_count += 1
                            if using_processed and row['original_path'] in foreign_lookup:
                                processed_count += 1
            except Exception as e:
                print(f"Error processing training row: {e}")
                continue

        print(f"  Training set composition:")
        print(f"    - Original images: {original_count} ({processed_count} from processing)")
        if using_augmented:
            print(f"    - Augmented images: {augmented_count}")
        print(f"    - Total: {len(X_train)}")

        # Prepare validation data
        print(f"\nPreparing validation data ({len(val_df)} rows)...")
        val_processed_count = 0

        for _, row in val_df.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "foreign")
                if img_array is not None:
                    X_val.append(img_array)
                    y_val.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in foreign_lookup:
                        val_processed_count += 1

        print(f"  Validation set: {len(X_val)} images ({val_processed_count} from processing)")

    # CASE 2: No train_df available - use foreign_metadata directly (non-augmented case)
    elif foreign_metadata is not None:
        print(f"\nPreparing training data from metadata ({len(foreign_metadata)} rows)...")
        train_processed_count = 0

        # Split the foreign data for training and validation
        from sklearn.model_selection import train_test_split

        # Extract all foreign data
        X_foreign = []
        y_foreign = []

        for _, row in foreign_metadata.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "foreign")
                if img_array is not None:
                    X_foreign.append(img_array)
                    y_foreign.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in foreign_lookup:
                        train_processed_count += 1

        print(f"  foreign dataset: {len(X_foreign)} images ({train_processed_count} from processing)")

        # Split into train and validation
        if len(X_foreign) > 0:
            X_train, X_val, y_train, y_val = train_test_split(
                X_foreign, y_foreign,
                test_size=0.2,
                random_state=42,
                stratify=y_foreign
            )
            print(f"  Training set: {len(X_train)} images")
            print(f"  Validation set: {len(X_val)} images")
        else:
            print("WARNING: No training data found!")

    else:
        print("WARNING: No training data source available!")

    # Prepare test data
    print(f"\nPreparing test data...")
    test_processed_count = 0

    # Try test_df first, then fall back to ph_metadata
    if test_df is not None:
        print(f"  From test_df ({len(test_df)} rows)...")
        for _, row in test_df.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "philippines")
                if img_array is not None:
                    X_test.append(img_array)
                    y_test.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in ph_lookup:
                        test_processed_count += 1
    elif ph_metadata is not None:
        print(f"  From ph_metadata ({len(ph_metadata)} rows)...")
        for _, row in ph_metadata.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "philippines")
                if img_array is not None:
                    X_test.append(img_array)
                    y_test.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in ph_lookup:
                        test_processed_count += 1

    print(f"  Test set: {len(X_test)} images ({test_processed_count} from processing)")

    # Convert to arrays
    X_train = safe_array_conversion(X_train)
    y_train = np.array(y_train)
    X_val = safe_array_conversion(X_val)
    y_val = np.array(y_val)
    X_test = safe_array_conversion(X_test)
    y_test = np.array(y_test)

    print(f"\nFinal dataset shapes:")
    print(f"Training: {X_train.shape}, {y_train.shape}")
    print(f"Validation: {X_val.shape}, {y_val.shape}")
    print(f"Test: {X_test.shape}, {y_test.shape}")

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
using_augmented = 'train_df' in locals() and 'is_augmented' in train_df.columns and any(train_df['is_augmented'])
using_processed = 'foreign_processed' in locals() and 'ph_processed' in locals() and foreign_processed is not None

print("="*60)
print("DATA SOURCE ANALYSIS")
print("="*60)
print(f"Augmented data available: {using_augmented}")
print(f"Processed images available: {using_processed}")

if using_augmented and using_processed:
    # CASE 1: BOTH augmented data AND processed images
    print("=== USING AUGMENTED DATA WITH PROCESSED IMAGES ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata, train_df, val_df, test_df,
        foreign_processed=foreign_processed, ph_processed=ph_processed
    )

elif using_augmented and not using_processed:
    # CASE 2: Only augmented data (no processed images)
    print("=== USING AUGMENTED DATA WITH ORIGINAL IMAGES ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata, train_df, val_df, test_df
    )

elif not using_augmented and using_processed:
    # CASE 3: Only processed images (no augmentation)
    print("=== USING PROCESSED IMAGES (NO AUGMENTATION) ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata,
        foreign_processed=foreign_processed, ph_processed=ph_processed
    )

else:
    # CASE 4: Neither - use original images only
    print("=== USING ORIGINAL IMAGES (NO AUGMENTATION) ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata
    )

In [ ]:
# Check dataset sizes and balance
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Training class distribution: {np.unique(y_train, return_counts=True)}")
print(f"Validation class distribution: {np.unique(y_val, return_counts=True)}")

# Check if data is shuffled properly
print(f"First 10 training labels: {y_train[:10]}")
print(f"First 10 validation labels: {y_val[:10]}")

# Check if shape is correct
print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Number of training samples: {len(X_train)}")
print(f"Number of validation samples: {len(X_val)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Steps per epoch: {len(X_train) // BATCH_SIZE}")

#### Debug

In [ ]:
def debug_training_speed(X_train, X_val, y_train, y_val):
    """Debug why training speed differs between augmented vs non-augmented"""
    print("\n" + "="*50)
    print("TRAINING SPEED DIAGNOSTICS")
    print("="*50)

    # Check data sources
    print(f"Training samples: {len(X_train)}")
    print(f"X_train type: {type(X_train)}")
    print(f"X_train dtype: {X_train.dtype}")
    print(f"X_train shape: {X_train.shape}")

    # Check if data is already normalized
    print(f"Data range: [{X_train.min():.3f}, {X_train.max():.3f}]")

    # Check memory usage
    import sys
    memory_mb = sys.getsizeof(X_train) / (1024 * 1024)
    print(f"Training data memory: {memory_mb:.2f} MB")

    # Check data loading source
    if hasattr(X_train, 'flags') and hasattr(X_train.flags, 'writeable'):
        print(f"Data is in memory: {X_train.flags.writeable}")

    return memory_mb

# Add this before training
memory_usage = debug_training_speed(X_train, X_val, y_train, y_val)

### 2.1 VGG16 Baseline Model

In [ ]:
# 2.1 VGG16 HYPERPARAMETER TUNING & FINE-TUNING
print("="*60)
print("2.1 VGG16 HYPERPARAMETER TUNING & FINE-TUNING")
print("="*60)

# =============================================================================
# 1. HYPERMODEL BUILDER FUNCTION
# =============================================================================
def build_hyper_vgg16(hp):
    """
    Builds a VGG16 hypermodel for KerasTuner.
    The base model (VGG16) is frozen for this stage.
    """
    print("Creating VGG16 hypermodel...")

    # 1. Base Model (VGG16)
    base_model = VGG16(
        include_top=False,
        weights='imagenet',
        input_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 3)
    )
    # Freeze the base model layers for initial training
    base_model.trainable = False

    # 2. Sequential Model
    model = models.Sequential()
    model.add(base_model)
    model.add(layers.Flatten())

    # --- Hyperparameter Tuning (Classifier Head) ---
    # Tune the number of units in the first dense layer
    hp_units_1 = hp.Int('dense_units_1', min_value=128, max_value=512, step=128)
    model.add(layers.Dense(units=hp_units_1, activation='relu'))

    # Tune the dropout rate after the first dense layer
    hp_dropout_1 = hp.Float('dropout_rate_1', min_value=0.3, max_value=0.7, step=0.2)
    model.add(layers.Dropout(rate=hp_dropout_1))

    # (Optional) Tune a second dense layer - let's keep it simple for now
    # hp_units_2 = hp.Int('dense_units_2', min_value=64, max_value=256, step=64)
    # model.add(layers.Dense(units=hp_units_2, activation='relu'))
    # hp_dropout_2 = hp.Float('dropout_rate_2', min_value=0.2, max_value=0.5, step=0.1)
    # model.add(layers.Dropout(rate=hp_dropout_2))
    # --- End Tuning ---

    model.add(layers.Dense(1, activation='sigmoid')) # Binary classification

    # 3. Compile
    # Tune the learning rate for the classifier head
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    print("✓ VGG16 hypermodel created successfully")
    return model

# =============================================================================
# 2. HYPERPARAMETER SEARCH (HEAD ONLY)
# =============================================================================
print("\nStarting KerasTuner Hyperband search...")

# Instantiate the tuner
tuner = kt.Hyperband(
    build_hyper_vgg16,
    objective='val_accuracy',
    max_epochs=20,  # Max epochs for *each* trial in the search
    factor=3,
    directory=os.path.join(OUTPUT_BASE, 'keras_tuner'),
    project_name='vgg16_head_tuning'
)

# Define callbacks for the search
# Note: We use a shorter patience for the search to speed it up
tuner_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

# Run the search using the in-memory data
# We don't need steps_per_epoch when providing numpy arrays
tuner.search(
    X_train, y_train,
    epochs=20, # This will be managed by Hyperband (max_epochs)
    validation_data=(X_val, y_val),
    callbacks=tuner_callbacks
)

# Get the best hyperparameters and the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
baseline_model = tuner.get_best_models(num_models=1)[0]

print("\n" + "="*50)
print(f"Hyperparameter Search Complete.")
print(f"Best Dense Units 1: {best_hps.get('dense_units_1')}")
print(f"Best Dropout Rate 1: {best_hps.get('dropout_rate_1')}")
print(f"Best Initial Learning Rate: {best_hps.get('learning_rate')}")
print("="*50 + "\n")

# Save the best model from the head-tuning phase
baseline_model.save(os.path.join(OUTPUT_BASE, 'baseline_vgg16_head_tuned.h5'))
print("Best head-only model saved.")
baseline_model.summary()

# =============================================================================
# 3. FINE-TUNING THE BEST MODEL
# =============================================================================
print("\nStarting fine-tuning of the best model...")

# 1. Unfreeze the VGG16 base model
# We need to access the VGG16 model *inside* the Sequential model
baseline_model.get_layer('vgg16').trainable = True

# 2. Freeze layers up to block5_conv1 (VGG16 has 19 layers)
# We will fine-tune from the 15th layer (block5_conv1) onwards
fine_tune_at = 15
for layer in baseline_model.get_layer('vgg16').layers[:fine_tune_at]:
    layer.trainable = False

print(f"Fine-tuning VGG16 model from layer {fine_tune_at} onwards.")

# 3. Re-compile the model with a VERY LOW learning rate
# This is crucial for fine-tuning to prevent destroying the pre-trained weights
FT_LEARNING_RATE = 1e-5

baseline_model.compile(
    optimizer=optimizers.Adam(learning_rate=FT_LEARNING_RATE), # Use the very low LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Model re-compiled with a fine-tuning learning rate of {FT_LEARNING_RATE}.")
baseline_model.summary() # Show the updated trainable params

# 4. Define callbacks for fine-tuning
# We can use a slightly longer patience here
ft_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7, verbose=1)
]

# 5. Continue training (fine-tuning)
print("\nStarting fine-tuning training...")
history_fine_tune = baseline_model.fit(
    X_train, y_train,
    epochs=50,  # Allow up to 50 epochs, EarlyStopping will handle the rest
    validation_data=(X_val, y_val),
    callbacks=ft_callbacks,
    verbose=1
)

print("✓ Fine-tuning complete.")

# =============================================================================
# 4. FINALIZING AND SAVING
# =============================================================================

# Evaluate final fine-tuned model
baseline_loss, baseline_accuracy = baseline_model.evaluate(X_val, y_val, verbose=0)

print(f"✓ Fine-Tuned VGG16 Training Completed")
print(f" Final Validation Accuracy: {baseline_accuracy:.4f}")
print(f" Final Validation Loss: {baseline_loss:.4f}")

# Save the final fine-tuned model as pickle
model_save_path = os.path.join(OUTPUT_BASE, 'baseline_vgg16_model.pkl')
joblib.dump(baseline_model, model_save_path)
print(f"✓ Final fine-tuned VGG16 model saved as: {model_save_path}")

# Save training history
history_save_path = os.path.join(OUTPUT_BASE, 'baseline_training_history.pkl')
joblib.dump(history_fine_tune.history, history_save_path)
print(f"✓ Fine-tuning history saved as: {history_save_path}")

# --- IMPORTANT ---
# Set variables for subsequent cells to use
# These MUST match the variable names your other cells expect
baseline_history = history_fine_tune
feature_extractor = baseline_model.get_layer('vgg16') # This is the VGG16 part
# 'baseline_model' is already correctly named
# 'baseline_accuracy' and 'baseline_loss' are also set

# Store baseline results
baseline_results = {
    'model': baseline_model,
    'history': baseline_history.history,
    'val_accuracy': baseline_accuracy,
    'val_loss': baseline_loss,
    'feature_extractor': feature_extractor
}

print("2.1 VGG16 Baseline Model - COMPLETED\n")

#### Model Summary

In [ ]:
def visualize_feature_maps(model, sample_image):
    vgg_base = model.layers[0]  # First layer is the VGG16 base

    print(f"VGG16 base type: {type(vgg_base)}")
    print(f"VGG16 base layers: {len(vgg_base.layers)}")

    feature_map_model = models.Model(inputs=vgg_base.input,
                                   outputs=vgg_base.output)

    feature_maps = feature_map_model.predict(sample_image)

    print(f"Feature maps shape: {feature_maps.shape}")

    fig, axes = plt.subplots(8, 8, figsize=(16, 16))
    fig.suptitle('7×7×512 Feature Maps from VGG16 (First 64 Channels)', fontsize=16, fontweight='bold')

    for i in range(8):
        for j in range(8):
            channel = i * 8 + j
            if channel < 64:
                ax = axes[i, j]
                ax.imshow(feature_maps[0, :, :, channel], cmap='viridis')
                ax.set_title(f'Channel {channel}', fontsize=8)
                ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'vgg16_feature_maps_7x7x512.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Also show the actual 7×7×512 dimensions
    print(f"\nFeature Map Dimensions Breakdown:")
    print(f"- Batch size: {feature_maps.shape[0]}")
    print(f"- Spatial dimensions: {feature_maps.shape[1]}×{feature_maps.shape[2]}")
    print(f"- Number of filters/channels: {feature_maps.shape[3]}")
    print(f"- Total elements: {feature_maps.shape[1]}×{feature_maps.shape[2]}×{feature_maps.shape[3]} = {feature_maps.shape[1]*feature_maps.shape[2]*feature_maps.shape[3]:,}")

    return feature_maps

sample_image = X_train[0:1] 
print(f"Sample image shape: {sample_image.shape}")

sample_label = y_train[0]
label_name = "LEAFBLAST" if sample_label == 1 else "HEALTHY"
print(f"Sample image label: {label_name}")

feature_maps = visualize_feature_maps(baseline_model, sample_image)

In [ ]:
#################
# 2025 Deepseek #
#################
def get_model_summary_detailed(model):
    """
    Display comprehensive model architecture
    """
    print("="*80)
    print("VGG16 BASELINE MODEL - COMPLETE ARCHITECTURE")
    print("="*80)

    # Standard summary
    model.summary()

    # Additional details about VGG16 base
    vgg_base = model.layers[0]
    print(f"\nVGG16 Base Details:")
    print(f"- Total layers in VGG16: {len(vgg_base.layers)}")
    print(f"- Input shape: {vgg_base.input_shape}")
    print(f"- Output shape: {vgg_base.output_shape}")  # This should be (None, 7, 7, 512)

    # Parameter counts
    total_params = model.count_params()
    trainable_count = np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    non_trainable_count = total_params - trainable_count

    print(f"\nParameter Summary:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_count:,} ({trainable_count/total_params*100:.1f}%)")
    print(f"Non-trainable parameters: {non_trainable_count:,} ({non_trainable_count/total_params*100:.1f}%)")

    return total_params, trainable_count, non_trainable_count

# Get the model summary
total_params, trainable_params, non_trainable_params = get_model_summary_detailed(baseline_model)

In [ ]:
#################
# 2025 Deepseek #
#################
def extract_features_correct(baseline_model, x_data):
    """
    Extracts features from the baseline VGG16 model.

    This function accesses the first layer (index 0) of the sequential
    'baseline_model', which IS the VGG16 feature extractor base model,
    and calls .predict() on it directly.
    """
    print(f"Extracting features from {x_data.shape[0]} images...")

    # The 'baseline_model' is Sequential, and its first layer (index 0)
    # is the VGG16 base model itself. We can just use it directly.
    feature_extractor_model = baseline_model.layers[0]

    # Predict to get the features
    print(f"Using model layer: {feature_extractor_model.name}")
    features = feature_extractor_model.predict(x_data, batch_size=BATCH_SIZE, verbose=1)

    print(f"✓ Features extracted, shape: {features.shape}")
    return features

print("Feature extraction function 'extract_features_correct' defined.")

In [ ]:
#################
# 2025 Deepseek #
#################
def extract_and_cache_baseline_features(baseline_model, X_train, X_val, X_test):
    """
    Extract features once from baseline model and cache them for PLSR/XGBoost
    This ensures ALL models use the EXACT same features
    """
    print("="*60)
    print("EXTRACTING & CACHING BASELINE FEATURES FOR ALL MODELS")
    print("="*60)
    
    # Extract features using the Flatten layer output
    print("Extracting training features...")
    train_features = extract_features_correct(baseline_model, X_train)
    
    print("Extracting validation features...") 
    val_features = extract_features_correct(baseline_model, X_val)
    
    print("Extracting test features...")
    test_features = extract_features_correct(baseline_model, X_test)
    
    # Ensure features are flattened (7x7x512 -> 25088)
    if len(train_features.shape) > 2:
        train_features = train_features.reshape(train_features.shape[0], -1)
        val_features = val_features.reshape(val_features.shape[0], -1) 
        test_features = test_features.reshape(test_features.shape[0], -1)
    
    print(f"Feature shapes - Train: {train_features.shape}, Val: {val_features.shape}, Test: {test_features.shape}")
    
    # Normalize features (important for PLSR/XGBoost)
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)
    test_features_scaled = scaler.transform(test_features)
    
    # Cache the features
    cached_features = {
        'train_raw': train_features,
        'val_raw': val_features, 
        'test_raw': test_features,
        'train_scaled': train_features_scaled,
        'val_scaled': val_features_scaled,
        'test_scaled': test_features_scaled,
        'scaler': scaler
    }
    
    # Save cached features
    joblib.dump(cached_features, os.path.join(OUTPUT_BASE, 'cached_baseline_features.pkl'))
    print("✓ Baseline features cached for PLSR/XGBoost reuse")
    
    return cached_features

# CALL THE FUNCTION
cached_features = extract_and_cache_baseline_features(baseline_model, X_train, X_val, X_test)

### 2.2 VGG16 + PLSR with Thresholding

In [ ]:
def train_plsr_model_with_cached_features(cached_features, y_train, y_val):
    print("Training PLSR with CACHED baseline features...")
    start_time = time.time()
    
    # Use the pre-extracted, pre-scaled features
    train_features_scaled = cached_features['train_scaled']
    val_features_scaled = cached_features['val_scaled']
    
    # Train PLSR
    plsr = PLSRegression(n_components=50, max_iter=1000)
    plsr.fit(train_features_scaled, y_train)

    training_time = time.time() - start_time
    print(f"✓ PLSR training completed in {training_time:.2f} seconds")

    # Predict
    y_pred_proba_plsr = plsr.predict(val_features_scaled)
    y_pred_plsr = (y_pred_proba_plsr > 0.5).astype(int).flatten()

    plsr_accuracy = accuracy_score(y_val, y_pred_plsr)
    plsr_precision = precision_score(y_val, y_pred_plsr, zero_division=0)
    plsr_recall = recall_score(y_val, y_pred_plsr, zero_division=0)
    plsr_f1 = f1_score(y_val, y_pred_plsr, zero_division=0)

    print(f"✓ PLSR Model Evaluation:")
    print(f" Validation Accuracy: {plsr_accuracy:.4f}")
    print(f" Precision: {plsr_precision:.4f}")
    print(f" Recall: {plsr_recall:.4f}")
    print(f" F1-Score: {plsr_f1:.4f}")

    # Save PLSR model
    plsr_model_path = os.path.join(OUTPUT_BASE, 'plsr_model.pkl')
    joblib.dump(plsr, plsr_model_path)
    print(f"✓ PLSR model saved as: {plsr_model_path}")

    return plsr, plsr_accuracy, plsr_precision, plsr_recall, plsr_f1

# Train PLSR model WITH CACHED FEATURES
plsr_model, plsr_accuracy, plsr_precision, plsr_recall, plsr_f1 = train_plsr_model_with_cached_features(
    cached_features, y_train, y_val
)

# Store PLSR results
plsr_results = {
    'model': plsr_model,
    'val_accuracy': plsr_accuracy,
    'precision': plsr_precision,
    'recall': plsr_recall,
    'f1_score': plsr_f1,
    'feature_extractor': baseline_model
}

print("2.2 VGG16 + PLSR with Thresholding - COMPLETED\n")

### 2.3 VGG16 + XGBoost Classifier

In [ ]:
def train_xgboost_model_with_cached_features(cached_features, y_train, y_val):
    if not XGB_AVAILABLE:
        print("XGBoost not available. Skipping...")
        return None, 0, 0, 0, 0, {}

    print("Training XGBoost with CACHED baseline features...")
    start_time = time.time()
    
    # Use pre-extracted, pre-scaled features
    train_features_scaled = cached_features['train_scaled']
    val_features_scaled = cached_features['val_scaled']
    
    # Train with evaluation history tracking
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        objective='binary:logistic',
        random_state=42,
        eval_metric='logloss'
    )
    
    # Train and capture history
    eval_set = [(train_features_scaled, y_train), (val_features_scaled, y_val)]
    xgb_model.fit(
        train_features_scaled, y_train,
        eval_set=eval_set,
        verbose=True
    )
    
    # Get training history
    xgb_history = xgb_model.evals_result()
    
    training_time = time.time() - start_time
    print(f"✓ XGBoost training completed in {training_time:.2f} seconds")

    # Predict
    y_pred_xgb = xgb_model.predict(val_features_scaled)
    y_pred_proba_xgb = xgb_model.predict_proba(val_features_scaled)[:, 1]

    xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
    xgb_precision = precision_score(y_val, y_pred_xgb, zero_division=0)
    xgb_recall = recall_score(y_val, y_pred_xgb, zero_division=0)
    xgb_f1 = f1_score(y_val, y_pred_xgb, zero_division=0)

    print(f"✓ XGBoost Model Evaluation:")
    print(f" Validation Accuracy: {xgb_accuracy:.4f}")
    print(f" Precision: {xgb_precision:.4f}")
    print(f" Recall: {xgb_recall:.4f}")
    print(f" F1-Score: {xgb_f1:.4f}")

    # Save XGBoost model
    if xgb_model is not None:
        xgb_model_path = os.path.join(OUTPUT_BASE, 'xgboost_model.pkl')
        joblib.dump(xgb_model, xgb_model_path)
        print(f"✓ XGBoost model saved as: {xgb_model_path}")

    return xgb_model, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1, xgb_history

# Train XGBoost model WITH CACHED FEATURES
xgb_model, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1, xgb_history = train_xgboost_model_with_cached_features(
    cached_features, y_train, y_val
)

# Store XGBoost results
xgb_results = {
    'model': xgb_model,
    'val_accuracy': xgb_accuracy,
    'precision': xgb_precision,
    'recall': xgb_recall,
    'f1_score': xgb_f1,
    'history': xgb_history,
    'feature_extractor': baseline_model
}

print("2.3 VGG16 + XGBoost - COMPLETED\n")

## 3. Validation Module

### 3.1 Model run with Validation set

In [ ]:
#################
# 2025 Deepseek #
#################
def validate_all_models(baseline_model, plsr_model, xgb_model, X_val, y_val, cached_features):
    """Comprehensive validation of all trained models USING CACHED FEATURES"""
    print("="*60)
    print("VALIDATION MODULE - MODEL EVALUATION")
    print("="*60)

    validation_results = {}

    # 1. Baseline VGG16 Validation
    print("\n1. Validating Baseline VGG16 Model...")
    baseline_val_proba = baseline_model.predict(X_val, verbose=0)
    baseline_val_pred = (baseline_val_proba > 0.5).astype(int).flatten()

    baseline_metrics = {
        'accuracy': accuracy_score(y_val, baseline_val_pred),
        'precision': precision_score(y_val, baseline_val_pred, zero_division=0),
        'recall': recall_score(y_val, baseline_val_pred, zero_division=0),
        'f1_score': f1_score(y_val, baseline_val_pred, zero_division=0),
        'predictions': baseline_val_pred,
        'probabilities': baseline_val_proba.flatten()
    }

    # Calculate specificity
    tn, fp, fn, tp = confusion_matrix(y_val, baseline_val_pred).ravel()
    baseline_metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

    validation_results['baseline'] = baseline_metrics

    print(f"  ✓ Accuracy: {baseline_metrics['accuracy']:.4f}")
    print(f"  ✓ Precision: {baseline_metrics['precision']:.4f}")
    print(f"  ✓ Recall: {baseline_metrics['recall']:.4f}")
    print(f"  ✓ F1-Score: {baseline_metrics['f1_score']:.4f}")
    print(f"  ✓ Specificity: {baseline_metrics['specificity']:.4f}")

    # 2. PLSR Model Validation - USING CACHED FEATURES
    print("\n2. Validating PLSR Model...")
    try:
        # Use cached validation features
        val_features_plsr_scaled = cached_features['val_scaled']
        plsr_val_pred_proba = plsr_model.predict(val_features_plsr_scaled)
        plsr_val_pred = (plsr_val_pred_proba > 0.5).astype(int).flatten()

        plsr_metrics = {
            'accuracy': accuracy_score(y_val, plsr_val_pred),
            'precision': precision_score(y_val, plsr_val_pred, zero_division=0),
            'recall': recall_score(y_val, plsr_val_pred, zero_division=0),
            'f1_score': f1_score(y_val, plsr_val_pred, zero_division=0),
            'predictions': plsr_val_pred,
            'probabilities': plsr_val_pred_proba.flatten()
        }

        tn, fp, fn, tp = confusion_matrix(y_val, plsr_val_pred).ravel()
        plsr_metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

        validation_results['plsr'] = plsr_metrics

        print(f"  ✓ Accuracy: {plsr_metrics['accuracy']:.4f}")
        print(f"  ✓ Precision: {plsr_metrics['precision']:.4f}")
        print(f"  ✓ Recall: {plsr_metrics['recall']:.4f}")
        print(f"  ✓ F1-Score: {plsr_metrics['f1_score']:.4f}")
        print(f"  ✓ Specificity: {plsr_metrics['specificity']:.4f}")
        
    except Exception as e:
        print(f"  ✗ PLSR validation failed: {e}")

    # 3. XGBoost Model Validation - USING CACHED FEATURES
    if XGB_AVAILABLE and xgb_model is not None:
        print("\n3. Validating XGBoost Model...")
        try:
            # Use cached validation features
            val_features_xgb_scaled = cached_features['val_scaled']
            xgb_val_pred = xgb_model.predict(val_features_xgb_scaled)
            xgb_val_pred_proba = xgb_model.predict_proba(val_features_xgb_scaled)[:, 1]

            xgb_metrics = {
                'accuracy': accuracy_score(y_val, xgb_val_pred),
                'precision': precision_score(y_val, xgb_val_pred, zero_division=0),
                'recall': recall_score(y_val, xgb_val_pred, zero_division=0),
                'f1_score': f1_score(y_val, xgb_val_pred, zero_division=0),
                'predictions': xgb_val_pred,
                'probabilities': xgb_val_pred_proba
            }

            tn, fp, fn, tp = confusion_matrix(y_val, xgb_val_pred).ravel()
            xgb_metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

            validation_results['xgboost'] = xgb_metrics

            print(f"  ✓ Accuracy: {xgb_metrics['accuracy']:.4f}")
            print(f"  ✓ Precision: {xgb_metrics['precision']:.4f}")
            print(f"  ✓ Recall: {xgb_metrics['recall']:.4f}")
            print(f"  ✓ F1-Score: {xgb_metrics['f1_score']:.4f}")
            print(f"  ✓ Specificity: {xgb_metrics['specificity']:.4f}")
            
        except Exception as e:
            print(f"  ✗ XGBoost validation failed: {e}")

    # Create validation results visualization
    plot_validation_results(validation_results, y_val)

    # Save validation results as pickle
    validation_results_path = os.path.join(OUTPUT_BASE, 'validation_results.pkl')
    joblib.dump(validation_results, validation_results_path)
    print(f"✓ Validation results saved as: {validation_results_path}")

    return validation_results

def plot_validation_results(validation_results, y_val):
    """Plot comprehensive validation results"""
    models = list(validation_results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'specificity']

    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()

    # Plot 1: All metrics comparison
    x = np.arange(len(models))
    width = 0.15

    for i, metric in enumerate(metrics):
        values = [validation_results[model][metric] for model in models]
        axes[0].bar(x + (i-2)*width, values, width, label=metric.capitalize(), alpha=0.8)

    axes[0].set_xlabel('Models')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Validation Metrics Comparison')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([model.upper() for model in models])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(0, 1)

    # Plot 2: Confusion matrices
    for idx, model in enumerate(models[:3]):  # Show first 3 models
        if idx < 3:  # Ensure we don't exceed subplot count
            cm = confusion_matrix(y_val, validation_results[model]['predictions'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx+1],
                       xticklabels=['Healthy', 'Blast'],
                       yticklabels=['Healthy', 'Blast'])
            axes[idx+1].set_title(f'{model.upper()} Confusion Matrix')
            axes[idx+1].set_xlabel('Predicted')
            axes[idx+1].set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'validation_results_comprehensive.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Print detailed classification reports
    print("\n" + "="*50)
    print("DETAILED CLASSIFICATION REPORTS")
    print("="*50)

    for model in models:
        print(f"\n{model.upper()} Classification Report:")
        print(classification_report(y_val, validation_results[model]['predictions'],
                                  target_names=['Healthy', 'Leaf Blast']))

# Run validation WITH CACHED FEATURES
print("Starting comprehensive model validation...")
validation_results = validate_all_models(baseline_model, plsr_model, xgb_model, X_val, y_val, cached_features)
print("Validation module completed successfully!")

### 3.2 Explainability Analysis

In [ ]:
#################
# 2025 Deepseek #
#################
def enhanced_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    try:
        # 1. Isolate the VGG16 base (First layer of your Sequential model)
        vgg_base = model.layers[0]

        # 2. Auto-detect the last CONVOLUTIONAL layer (skipping Pooling layers)
        if last_conv_layer_name is None:
            for layer in reversed(vgg_base.layers):
                # Check if it is a Conv2D layer (has 4D output and weights)
                if 'conv' in layer.name or isinstance(layer, tf.keras.layers.Conv2D):
                    last_conv_layer_name = layer.name
                    break

        # 3. Construct a sub-model for the VGG base
        # We need TWO outputs: The specific Conv layer (for heat) and the Final Base output (for flow)
        base_multi_out_model = tf.keras.models.Model(
            inputs=vgg_base.input,
            outputs=[vgg_base.get_layer(last_conv_layer_name).output, vgg_base.output]
        )

        # 4. Run the Gradient Calculation
        with tf.GradientTape() as tape:
            # Get the conv features and the base output
            conv_outputs, base_outputs = base_multi_out_model(img_array)

            # Manually pass the base output through the rest of your classifier (The "Head")
            # This connects the VGG base to your final Dense prediction
            preds = base_outputs
            for layer in model.layers[1:]: # Skip the first layer (vgg_base)
                preds = layer(preds)

            # Get the score for the top predicted class
            top_pred_index = tf.argmax(preds[0])
            loss = preds[:, top_pred_index]

        # 5. Calculate Gradients
        grads = tape.gradient(loss, conv_outputs)

        # 6. Pool Gradients (GAP)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        # 7. Generate Heatmap
        conv_outputs = conv_outputs[0]
        heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

        # 8. Clean Up Heatmap
        heatmap = heatmap.numpy()
        heatmap = np.maximum(heatmap, 0) # ReLU (Remove negative values)

        # CRITICAL: Noise Gate to "Specificially Highlight Blast"
        # If the heatmap is too fuzzy, we zero out weak activations
        if np.max(heatmap) > 0:
            heatmap /= np.max(heatmap) # Normalize 0-1

            # Optional: Threshold to remove background noise (e.g., weak activations < 0.4)
            # This makes the "spots" pop out more
            heatmap[heatmap < 0.3] = 0

        return heatmap

    except Exception as e:
        print(f"Grad-CAM Error: {e}")
        return improved_fallback_heatmap(img_array, model)

def improved_fallback_heatmap(img_array, model):
    """Fallback to edge detection if Grad-CAM completely fails"""
    try:
        img_uint8 = (img_array[0] * 255).astype(np.uint8)
        img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        grad_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=5)
        grad_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=5)
        magnitude = np.sqrt(grad_x**2 + grad_y**2)
        magnitude = cv2.GaussianBlur(magnitude, (15, 15), 3)
        heatmap = cv2.resize(magnitude, (7, 7))
        heatmap = np.power(heatmap, 0.6)
        if np.max(heatmap) > 0: heatmap /= np.max(heatmap)
        return heatmap
    except:
        return np.zeros((7, 7))

def apply_gradcam_validation(X_val, y_val, model, n_samples=6):
    """Apply Grad-CAM to validation samples and display results"""
    print("Applying Grad-CAM to validation samples...")

    # Pick random samples
    indices = np.random.choice(len(X_val), min(n_samples, len(X_val)), replace=False)

    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}

    # Setup Plot
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5 * n_samples))
    if n_samples == 1: axes = np.expand_dims(axes, axis=0)

    for row_idx, idx in enumerate(indices):
        img_array_sample = X_val[idx:idx+1]
        actual_label = y_val[idx]

        # Prediction
        pred_proba = model.predict(img_array_sample, verbose=0)[0][0]
        pred_class = 1 if pred_proba > 0.5 else 0

        # Generate Heatmap
        heatmap = enhanced_gradcam_heatmap(img_array_sample, model)

        # Resize heatmap to match image size (224x224)
        heatmap_resized = cv2.resize(heatmap, (TARGET_SIZE[0], TARGET_SIZE[1]))

        # Prepare Image
        # Assuming X_val is 0-1 float, convert to 0-255 uint8
        original_img = (img_array_sample[0] * 255).astype(np.uint8)

        # Create Overlay
        # Convert heatmap to RGB (Jet colormap)
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        # Blend: 60% Original + 40% Heatmap
        superimposed = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)

        # Display
        # Column 1: Original
        axes[row_idx, 0].imshow(original_img)
        axes[row_idx, 0].set_title(f'Actual: {class_names[actual_label]}', fontsize=12)
        axes[row_idx, 0].axis('off')

        # Column 2: Heatmap Only
        axes[row_idx, 1].imshow(heatmap_resized, cmap='jet')
        axes[row_idx, 1].set_title('Model Attention', fontsize=12)
        axes[row_idx, 1].axis('off')

        # Column 3: Overlay
        axes[row_idx, 2].imshow(superimposed)
        axes[row_idx, 2].set_title(f'Pred: {class_names[pred_class]} ({pred_proba:.2f})', fontsize=12)
        axes[row_idx, 2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'gradcam_validation_final.png'), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
#################
# 2025 Deepseek #
#################
def comparative_explainability_validation(baseline_model, plsr_model, xgb_model, X_val, y_val, cached_features, n_samples=6):
    """Compare all three models on validation set USING CACHED FEATURES"""
    print("="*60)
    print("COMPARATIVE EXPLAINABILITY - VALIDATION SET")
    print("="*60)

    # Select validation samples
    indices = np.random.choice(len(X_val), min(n_samples, len(X_val)), replace=False)

    fig, axes = plt.subplots(n_samples, 4, figsize=(18, 4 * n_samples))
    if n_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}

    for row_idx, idx in enumerate(indices):
        img_array = X_val[idx:idx+1]
        actual_label = y_val[idx]
        actual_class = class_names[actual_label]

        # Get predictions from ALL models
        baseline_pred = baseline_model.predict(img_array, verbose=0)[0][0]
        plsr_pred = get_plsr_prediction_with_cached(baseline_model, plsr_model, cached_features, img_array)
        xgb_pred = get_xgb_prediction_with_cached(baseline_model, xgb_model, cached_features, img_array)

        # Generate attention maps
        baseline_heatmap = enhanced_gradcam_heatmap(img_array, baseline_model)
        plsr_heatmap = generate_plsr_attention_with_cached(baseline_model, plsr_model, cached_features, img_array)
        xgb_heatmap = generate_xgb_attention_with_cached(baseline_model, xgb_model, cached_features, img_array)

        # Original image
        axes[row_idx, 0].imshow(img_array[0])
        axes[row_idx, 0].set_title(f'Actual: {actual_class}', fontsize=12, fontweight='bold')
        axes[row_idx, 0].axis('off')

        # Baseline VGG16
        plot_model_attention_validation(axes[row_idx, 1], img_array[0], baseline_heatmap,
                                      baseline_pred, 'VGG16 Baseline', class_names)

        # VGG16 + PLSR
        plot_model_attention_validation(axes[row_idx, 2], img_array[0], plsr_heatmap,
                                      plsr_pred, 'VGG16 + PLSR', class_names)

        # VGG16 + XGBoost
        plot_model_attention_validation(axes[row_idx, 3], img_array[0], xgb_heatmap,
                                      xgb_pred, 'VGG16 + XGBoost', class_names)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comparative_explainability_validation.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# UPDATED HELPER FUNCTIONS THAT USE CACHED FEATURES
def get_plsr_prediction_with_cached(baseline_model, plsr_model, cached_features, img_array):
    """Get PLSR prediction for a single image USING CACHED FEATURE PROCESSING"""
    features = extract_features_correct(baseline_model, img_array)
    if len(features.shape) > 2:
        features = features.reshape(features.shape[0], -1)
    
    # Use the same scaler from cached features
    features_scaled = cached_features['scaler'].transform(features)
    return plsr_model.predict(features_scaled)[0][0]

def get_xgb_prediction_with_cached(baseline_model, xgb_model, cached_features, img_array):
    """Get XGBoost prediction for a single image USING CACHED FEATURE PROCESSING"""
    if not XGB_AVAILABLE or xgb_model is None:
        return 0.5
    
    features = extract_features_correct(baseline_model, img_array)
    if len(features.shape) > 2:
        features = features.reshape(features.shape[0], -1)
    
    # Use the same scaler from cached features
    features_scaled = cached_features['scaler'].transform(features)
    return xgb_model.predict_proba(features_scaled)[0][1]

def generate_plsr_attention_with_cached(baseline_model, plsr_model, cached_features, img_array):
    """Create attention map showing what PLSR focuses on USING CACHED FEATURES"""
    features = extract_features_correct(baseline_model, img_array)

    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)

        # Get PLSR coefficients and project back to spatial dimensions
        plsr_coef = np.abs(plsr_model.coef_.flatten())

        # Handle shape mismatch by taking the first n features
        n_features = min(plsr_coef.shape[0], features_flat.shape[1])
        spatial_weights = plsr_coef[:n_features].reshape(original_shape[0], original_shape[1], -1)
        spatial_weights = spatial_weights.mean(axis=2)  # Average across channels
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))

    # Resize to match original image
    attention_map = cv2.resize(spatial_weights, (224, 224))

    # Apply smoothing and normalization
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)

    return attention_map

def generate_xgb_attention_with_cached(baseline_model, xgb_model, cached_features, img_array):
    """Create attention map showing what XGBoost focuses on USING CACHED FEATURES"""
    if not XGB_AVAILABLE or xgb_model is None:
        return np.zeros((224, 224))

    features = extract_features_correct(baseline_model, img_array)

    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)

        # Get XGBoost feature importances
        if hasattr(xgb_model, 'feature_importances_'):
            xgb_importance = xgb_model.feature_importances_
            n_features = min(xgb_importance.shape[0], features_flat.shape[1])
            spatial_weights = xgb_importance[:n_features].reshape(original_shape[0], original_shape[1], -1)
            spatial_weights = spatial_weights.mean(axis=2)
        else:
            # Fallback: use uniform weights
            spatial_weights = np.ones((original_shape[0], original_shape[1]))
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))

    # Resize and process
    attention_map = cv2.resize(spatial_weights, (224, 224))
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)

    return attention_map

# Keep this function as is (it doesn't use scalers)
def plot_model_attention_validation(ax, original_img, heatmap, prediction, model_name, class_names):
    """Plot individual model attention for validation set"""
    pred_class = 1 if prediction > 0.5 else 0
    pred_label = class_names[pred_class]
    confidence = prediction if pred_class == 1 else 1 - prediction

    original_uint8 = (original_img * 255).astype(np.uint8)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(original_uint8, 0.6, heatmap_colored, 0.4, 0)

    ax.imshow(superimposed)
    ax.set_title(f'{model_name}\nPred: {pred_label}\n({confidence:.3f})', fontsize=10)
    ax.axis('off')

# Call it with cached features
print("Running comparative explainability on validation set...")
comparative_explainability_validation(baseline_model, plsr_model, xgb_model, X_val, y_val, cached_features)

## 4. Model Deployment Module

### 4.1 Optimal Model run with Testing set

In [ ]:
#################
# 2025 Deepseek #
#################
def prepare_test_data(ph_processed):
    """Prepare test data from the processed Philippines dataset"""
    print("Preparing test data from processed images...")

    X_test = []
    y_test = []
    test_info = []

    for img_data in ph_processed:
        # Use the already processed image from background removal
        processed_img = img_data['processed_image']
        label = auto_detect_label(img_data['original_path'])
        label_val = 1 if label == 'LEAFBLAST' else 0

        if processed_img is not None:
            X_test.append(processed_img)
            y_test.append(label_val)
            test_info.append({
                'original_path': img_data['original_path'],
                'annotated_name': create_annotation_format(
                    Path(img_data['original_path']).name,
                    label,
                    'LOCAL',
                    len(test_info) + 1
                ),
                'label': label,
                'label_numeric': label_val
            })

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    print(f"Test data prepared: {X_test.shape[0]} images")
    return X_test, y_test, test_info

def evaluate_model_testing(model, model_type, X_test, y_test, feature_extractor=None, cached_features=None):
    """Comprehensive model evaluation on test set USING CACHED FEATURES"""
    start_time = time.time()

    if model_type == 'cnn':
        # CNN model (Baseline VGG16)
        y_pred_proba = model.predict(X_test, verbose=0)
        y_pred = (y_pred_proba > 0.5).astype(int).flatten()
        inference_time = time.time() - start_time

    elif model_type in ['plsr', 'xgboost']:
        # Feature-based models - USE CACHED FEATURES
        if feature_extractor is None or cached_features is None:
            raise ValueError("Feature extractor and cached_features required for feature-based models")

        # Use pre-extracted, pre-scaled test features
        test_features_scaled = cached_features['test_scaled']

        if model_type == 'plsr':
            y_pred_proba = model.predict(test_features_scaled)
            y_pred = (y_pred_proba > 0.5).astype(int).flatten()
        else:  # xgboost
            y_pred = model.predict(test_features_scaled)
            y_pred_proba = model.predict_proba(test_features_scaled)[:, 1]

        inference_time = time.time() - start_time

    # Calculate comprehensive metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Specificity (True Negative Rate)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'specificity': specificity,
        'inference_time': inference_time,
        'predictions': y_pred,
        'probabilities': y_pred_proba,
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }

    return results

def deploy_all_models_testing(baseline_model, plsr_model, xgb_model, test_df, cached_features):
    """Deploy all models on test set and compare performance USING CACHED FEATURES"""
    print("="*60)
    print("MODEL DEPLOYMENT - TEST SET EVALUATION (WITH CACHED FEATURES)")
    print("="*60)

    # Use the SAME test set that was used for caching (to avoid size mismatch)
    print("Using cached test features to ensure consistency...")
    
    # The cached features already contain the correct test set
    # We need to get the actual test images that correspond to the cached features
    X_test_cached, y_test_cached, test_info_cached = get_test_set_from_cache(cached_features, ph_processed)
    
    test_results = {}

    # 1. Evaluate Baseline VGG16
    print("\n1. Testing Baseline VGG16 Model...")
    baseline_test_results = evaluate_model_testing(
        baseline_model, 'cnn', X_test_cached, y_test_cached
    )
    test_results['baseline'] = baseline_test_results

    print(f"  ✓ Accuracy: {baseline_test_results['accuracy']:.4f}")
    print(f"  ✓ Precision: {baseline_test_results['precision']:.4f}")
    print(f"  ✓ Recall: {baseline_test_results['recall']:.4f}")
    print(f"  ✓ F1-Score: {baseline_test_results['f1_score']:.4f}")
    print(f"  ✓ Specificity: {baseline_test_results['specificity']:.4f}")
    print(f"  ✓ Inference Time: {baseline_test_results['inference_time']:.2f}s")

    # 2. Evaluate PLSR Model - USING CACHED FEATURES
    print("\n2. Testing PLSR Model...")
    plsr_test_results = evaluate_model_testing(
        plsr_model, 'plsr', X_test_cached, y_test_cached,
        feature_extractor=baseline_model, cached_features=cached_features
    )
    test_results['plsr'] = plsr_test_results

    print(f"  ✓ Accuracy: {plsr_test_results['accuracy']:.4f}")
    print(f"  ✓ Precision: {plsr_test_results['precision']:.4f}")
    print(f"  ✓ Recall: {plsr_test_results['recall']:.4f}")
    print(f"  ✓ F1-Score: {plsr_test_results['f1_score']:.4f}")
    print(f"  ✓ Specificity: {plsr_test_results['specificity']:.4f}")
    print(f"  ✓ Inference Time: {plsr_test_results['inference_time']:.2f}s")

    # 3. Evaluate XGBoost Model - USING CACHED FEATURES
    if XGB_AVAILABLE and xgb_model is not None:
        print("\n3. Testing XGBoost Model...")
        xgb_test_results = evaluate_model_testing(
            xgb_model, 'xgboost', X_test_cached, y_test_cached,
            feature_extractor=baseline_model, cached_features=cached_features
        )
        test_results['xgboost'] = xgb_test_results

        print(f"  ✓ Accuracy: {xgb_test_results['accuracy']:.4f}")
        print(f"  ✓ Precision: {xgb_test_results['precision']:.4f}")
        print(f"  ✓ Recall: {xgb_test_results['recall']:.4f}")
        print(f"  ✓ F1-Score: {xgb_test_results['f1_score']:.4f}")
        print(f"  ✓ Specificity: {xgb_test_results['specificity']:.4f}")
        print(f"  ✓ Inference Time: {xgb_test_results['inference_time']:.2f}s")

    # Create comprehensive test results visualization
    plot_test_results_comprehensive(test_results, y_test_cached)

    # Save test results as pickle
    test_results_path = os.path.join(OUTPUT_BASE, 'test_results.pkl')
    joblib.dump(test_results, test_results_path)
    print(f"✓ Test results saved as: {test_results_path}")

    # Save test info
    test_info_path = os.path.join(OUTPUT_BASE, 'test_info.pkl')
    joblib.dump(test_info_cached, test_info_path)
    print(f"✓ Test info saved as: {test_info_path}")

    return test_results, X_test_cached, y_test_cached, test_info_cached

def get_test_set_from_cache(cached_features, ph_processed):
    """Get the test images that correspond to the cached features"""
    print("Matching cached features with original test images...")
    
    # We need to find which images were actually used in the cached test features
    # Since we can't directly match, we'll use the first n images from ph_processed
    # where n matches the number of cached test features
    
    n_cached_test = cached_features['test_scaled'].shape[0]
    print(f"Cached test features: {n_cached_test} samples")
    print(f"Available processed images: {len(ph_processed)} samples")
    
    # Use the first n_cached_test images from ph_processed
    X_test = []
    y_test = []
    test_info = []
    
    for i in range(min(n_cached_test, len(ph_processed))):
        img_data = ph_processed[i]
        processed_img = img_data['processed_image']
        label = auto_detect_label(img_data['original_path'])
        label_val = 1 if label == 'LEAFBLAST' else 0

        if processed_img is not None:
            X_test.append(processed_img)
            y_test.append(label_val)
            test_info.append({
                'original_path': img_data['original_path'],
                'annotated_name': create_annotation_format(
                    Path(img_data['original_path']).name,
                    label,
                    'LOCAL',
                    i + 1
                ),
                'label': label,
                'label_numeric': label_val
            })
    
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    
    print(f"Matched test set: {len(X_test)} images")
    
    # Verify the match
    if len(X_test) != n_cached_test:
        print(f"⚠️  WARNING: Test set size mismatch! Using {len(X_test)} images instead of {n_cached_test}")
    
    return X_test, y_test, test_info

def plot_test_results_comprehensive(test_results, y_test):
    """Create comprehensive visualization of test results - ALL MODELS IN ONE FIGURE"""
    models = list(test_results.keys())
    
    # Determine layout based on number of models
    n_models = len(models)
    
    if n_models == 1:
        # Single model layout
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        axes = axes.flatten()
    elif n_models == 2:
        # Two models layout  
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()
    else:
        # Three models layout
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()

    # Plot 1: Performance metrics comparison
    metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'specificity']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']

    x = np.arange(len(models))
    width = 0.15

    for i, metric in enumerate(metrics):
        values = [test_results[model][metric] for model in models]
        axes[0].bar(x + (i-2)*width, values, width, label=metric_labels[i], alpha=0.8)

    axes[0].set_xlabel('Models')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Test Set Performance Metrics', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([model.upper() for model in models])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(0, 1)

    # Plot 2: Inference time comparison
    axes[1].bar(models, [test_results[model]['inference_time'] for model in models], 
                color='orange', alpha=0.7)
    axes[1].set_xlabel('Models')
    axes[1].set_ylabel('Seconds')
    axes[1].set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    # Add value labels on bars
    for i, model in enumerate(models):
        time_val = test_results[model]['inference_time']
        axes[1].text(i, time_val + 0.01, f'{time_val:.2f}s', 
                    ha='center', va='bottom')

    # Plot 3+: Confusion matrices for available models
    for idx, model in enumerate(models):
        if idx < len(axes) - 2:  # Ensure we don't exceed available subplots
            cm = test_results[model]['confusion_matrix']
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx+2],
                       xticklabels=['Healthy', 'Blast'],
                       yticklabels=['Healthy', 'Blast'])
            axes[idx+2].set_title(f'{model.upper()} Confusion Matrix', fontsize=12, fontweight='bold')
            axes[idx+2].set_xlabel('Predicted')
            axes[idx+2].set_ylabel('Actual')

    # Hide any unused subplots
    for i in range(2 + len(models), len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'test_results_comprehensive_all_models.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# Run deployment testing
print("Starting model deployment on test set...")
test_results, X_test, y_test, test_info = deploy_all_models_testing(
    baseline_model, plsr_model, xgb_model, test_df, cached_features
)
print("Model deployment testing completed successfully!")

### 4.2 Explainability Analysis

In [ ]:
#################
# 2025 Deepseek #
#################
def comparative_explainability_test_set(baseline_model, plsr_model, xgb_model, X_test, y_test, test_info, cached_features, n_samples=8):
    """Comprehensive comparative explainability on test set USING CACHED FEATURES"""
    print("="*60)
    print("COMPARATIVE EXPLAINABILITY - TEST SET (WITH CACHED FEATURES)")
    print("="*60)

    # Select diverse test samples (correct/incorrect predictions)
    test_predictions = baseline_model.predict(X_test, verbose=0).flatten()
    test_pred_classes = (test_predictions > 0.5).astype(int)

    correct_indices = np.where(test_pred_classes == y_test)[0]
    incorrect_indices = np.where(test_pred_classes != y_test)[0]

    n_each = min(n_samples // 2, len(correct_indices), len(incorrect_indices))
    correct_sample = np.random.choice(correct_indices, n_each, replace=False)
    incorrect_sample = np.random.choice(incorrect_indices, n_each, replace=False)
    selected_indices = np.concatenate([correct_sample, incorrect_sample])

    fig, axes = plt.subplots(len(selected_indices), 4, figsize=(18, 4 * len(selected_indices)))
    if len(selected_indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}

    for row_idx, idx in enumerate(selected_indices):
        img_array = X_test[idx:idx+1]
        actual_label = y_test[idx]
        actual_class = class_names[actual_label]

        # Get ALL model predictions USING CACHED FEATURES
        baseline_pred = baseline_model.predict(img_array, verbose=0)[0][0]
        plsr_pred = get_plsr_prediction_with_cached(baseline_model, plsr_model, cached_features, img_array)
        xgb_pred = get_xgb_prediction_with_cached(baseline_model, xgb_model, cached_features, img_array)

        # Generate ALL attention maps USING CACHED FEATURES
        baseline_heatmap = enhanced_gradcam_heatmap(img_array, baseline_model)
        plsr_heatmap = generate_plsr_attention_with_cached(baseline_model, plsr_model, cached_features, img_array)
        xgb_heatmap = generate_xgb_attention_with_cached(baseline_model, xgb_model, cached_features, img_array)

        # Check if baseline prediction is correct
        baseline_correct = (baseline_pred > 0.5) == actual_label
        result_color = 'green' if baseline_correct else 'red'
        result_text = 'correct' if baseline_correct else 'wrong'

        # Original image with correctness indicator
        axes[row_idx, 0].imshow(img_array[0])
        axes[row_idx, 0].set_title(f'Actual: {actual_class}\n{result_text}',
                                 color=result_color, fontsize=12, fontweight='bold')
        axes[row_idx, 0].axis('off')

        # All three models
        plot_model_attention_test(axes[row_idx, 1], img_array[0], baseline_heatmap,
                                baseline_pred, 'VGG16 Baseline', class_names)
        plot_model_attention_test(axes[row_idx, 2], img_array[0], plsr_heatmap,
                                plsr_pred, 'VGG16 + PLSR', class_names)
        plot_model_attention_test(axes[row_idx, 3], img_array[0], xgb_heatmap,
                                xgb_pred, 'VGG16 + XGBoost', class_names)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comparative_explainability_test_set.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Run model agreement analysis USING CACHED FEATURES
    agreements, disagreements = analyze_model_agreement_with_cached(baseline_model, plsr_model, xgb_model, X_test, y_test, cached_features)

def plot_model_attention_test(ax, original_img, heatmap, prediction, model_name, class_names):
    """Plot individual model attention for test set with enhanced styling"""
    pred_class = 1 if prediction > 0.5 else 0
    pred_label = class_names[pred_class]
    confidence = prediction if pred_class == 1 else 1 - prediction

    original_uint8 = (original_img * 255).astype(np.uint8)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(original_uint8, 0.6, heatmap_colored, 0.4, 0)

    ax.imshow(superimposed)

    # Color code based on confidence
    confidence_color = 'green' if confidence > 0.7 else 'orange' if confidence > 0.5 else 'red'

    ax.set_title(f'{model_name}\n{pred_label}\nConf: {confidence:.3f}',
                fontsize=10, color=confidence_color, fontweight='bold')
    ax.axis('off')

# UPDATED HELPER FUNCTIONS THAT USE CACHED FEATURES
def get_plsr_prediction_with_cached(baseline_model, plsr_model, cached_features, img_array):
    """Get PLSR prediction for a single image USING CACHED FEATURE PROCESSING"""
    features = extract_features_correct(baseline_model, img_array)
    if len(features.shape) > 2:
        features = features.reshape(features.shape[0], -1)
    
    # Use the same scaler from cached features
    features_scaled = cached_features['scaler'].transform(features)
    return plsr_model.predict(features_scaled)[0][0]

def get_xgb_prediction_with_cached(baseline_model, xgb_model, cached_features, img_array):
    """Get XGBoost prediction for a single image USING CACHED FEATURE PROCESSING"""
    if not XGB_AVAILABLE or xgb_model is None:
        return 0.5
    
    features = extract_features_correct(baseline_model, img_array)
    if len(features.shape) > 2:
        features = features.reshape(features.shape[0], -1)
    
    # Use the same scaler from cached features
    features_scaled = cached_features['scaler'].transform(features)
    return xgb_model.predict_proba(features_scaled)[0][1]

def generate_plsr_attention_with_cached(baseline_model, plsr_model, cached_features, img_array):
    """Create attention map showing what PLSR focuses on USING CACHED FEATURES"""
    features = extract_features_correct(baseline_model, img_array)

    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)

        # Get PLSR coefficients and project back to spatial dimensions
        plsr_coef = np.abs(plsr_model.coef_.flatten())

        # Handle shape mismatch by taking the first n features
        n_features = min(plsr_coef.shape[0], features_flat.shape[1])
        spatial_weights = plsr_coef[:n_features].reshape(original_shape[0], original_shape[1], -1)
        spatial_weights = spatial_weights.mean(axis=2)  # Average across channels
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))

    # Resize to match original image
    attention_map = cv2.resize(spatial_weights, (224, 224))

    # Apply smoothing and normalization
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)

    return attention_map

def generate_xgb_attention_with_cached(baseline_model, xgb_model, cached_features, img_array):
    """Create attention map showing what XGBoost focuses on USING CACHED FEATURES"""
    if not XGB_AVAILABLE or xgb_model is None:
        return np.zeros((224, 224))

    features = extract_features_correct(baseline_model, img_array)

    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)

        # Get XGBoost feature importances
        if hasattr(xgb_model, 'feature_importances_'):
            xgb_importance = xgb_model.feature_importances_
            n_features = min(xgb_importance.shape[0], features_flat.shape[1])
            spatial_weights = xgb_importance[:n_features].reshape(original_shape[0], original_shape[1], -1)
            spatial_weights = spatial_weights.mean(axis=2)
        else:
            # Fallback: use uniform weights
            spatial_weights = np.ones((original_shape[0], original_shape[1]))
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))

    # Resize and process
    attention_map = cv2.resize(spatial_weights, (224, 224))
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)

    return attention_map

# UPDATED Model Agreement Analysis
def analyze_model_agreement_with_cached(baseline_model, plsr_model, xgb_model, X_test, y_test, cached_features):
    """Analyze where models agree/disagree USING CACHED FEATURES"""
    agreements = []
    disagreements = []

    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}

    for i in range(len(X_test)):
        img_array = X_test[i:i+1]
        actual = y_test[i]

        # Get all predictions USING CACHED FEATURES
        baseline_pred = baseline_model.predict(img_array, verbose=0)[0][0]
        plsr_pred = get_plsr_prediction_with_cached(baseline_model, plsr_model, cached_features, img_array)
        xgb_pred = get_xgb_prediction_with_cached(baseline_model, xgb_model, cached_features, img_array)

        pred_classes = [baseline_pred > 0.5, plsr_pred > 0.5, xgb_pred > 0.5]

        # Check agreement
        if len(set(pred_classes)) == 1:  # All models agree
            agreements.append(i)
        else:
            disagreements.append({
                'index': i, 'actual': actual, 'predictions': pred_classes,
                'confidence': [baseline_pred, plsr_pred, xgb_pred]
            })

    print(f"\nModel Agreement Analysis:")
    print(f"Total test samples: {len(X_test)}")
    print(f"Full agreement: {len(agreements)} ({len(agreements)/len(X_test)*100:.1f}%)")
    print(f"Disagreements: {len(disagreements)} ({len(disagreements)/len(X_test)*100:.1f}%)")

    # Show some disagreement cases
    if disagreements:
        print(f"\nShowing {min(3, len(disagreements))} disagreement cases:")
        for i, case in enumerate(disagreements[:3]):
            print(f"Case {i+1}: Actual={class_names[case['actual']]}, "
                  f"Preds=[VGG16:{case['predictions'][0]}, PLSR:{case['predictions'][1]}, XGB:{case['predictions'][2]}]")

    return agreements, disagreements

# Call it with cached features
print("Running enhanced comparative explainability on test set...")
comparative_explainability_test_set(baseline_model, plsr_model, xgb_model, X_test, y_test, test_info, cached_features)

## 5. Reporting Module

In [ ]:
#################
# 2025 Deepseek #
#################
def generate_comprehensive_report(validation_results, test_results, train_df, val_df, test_df):
    """Generate final comprehensive performance report"""
    print("="*60)
    print("COMPREHENSIVE PERFORMANCE REPORT")
    print("="*60)

    # Create results comparison dataframe
    report_data = []
    models = list(test_results.keys())

    for model in models:
        if model in validation_results and model in test_results:
            val_result = validation_results[model]
            test_result = test_results[model]

            report_data.append({
                'Model': model.upper(),
                'Val_Accuracy': val_result['accuracy'],
                'Test_Accuracy': test_result['accuracy'],
                'Test_Precision': test_result['precision'],
                'Test_Recall': test_result['recall'],
                'Test_F1_Score': test_result['f1_score'],
                'Test_Specificity': test_result['specificity'],  # Already included
                'Inference_Time_Seconds': test_result['inference_time'],
                'Performance_Gap': val_result['accuracy'] - test_result['accuracy']
            })

    report_df = pd.DataFrame(report_data)

    # Display the report table
    print("\nPERFORMANCE COMPARISON ACROSS MODELS:")
    print("="*50)
    print(report_df.round(4))

    # Dataset statistics
    print("\n" + "="*50)
    print("DATASET STATISTICS")
    print("="*50)

    dataset_stats = {
        'Dataset': ['Training (foreign)', 'Validation (foreign)', 'Testing (Philippines)'],
        'Total_Images': [len(train_df), len(val_df), len(test_df)],
        'LEAFBLAST_Count': [
            len(train_df[train_df['label'] == 'LEAFBLAST']),
            len(val_df[val_df['label'] == 'LEAFBLAST']),
            len(test_df[test_df['label'] == 'LEAFBLAST'])
        ],
        'HEALTHY_Count': [
            len(train_df[train_df['label'] == 'HEALTHY']),
            len(val_df[val_df['label'] == 'HEALTHY']),
            len(test_df[test_df['label'] == 'HEALTHY'])
        ],
        'LEAFBLAST_Percentage': [
            len(train_df[train_df['label'] == 'LEAFBLAST']) / len(train_df) * 100,
            len(val_df[val_df['label'] == 'LEAFBLAST']) / len(val_df) * 100,
            len(test_df[test_df['label'] == 'LEAFBLAST']) / len(test_df) * 100
        ]
    }

    stats_df = pd.DataFrame(dataset_stats)
    print(stats_df.round(2))

    # Model comparison analysis
    print("\n" + "="*50)
    print("MODEL COMPARISON ANALYSIS")
    print("="*50)

    best_test_accuracy = report_df['Test_Accuracy'].max()
    best_model = report_df.loc[report_df['Test_Accuracy'].idxmax(), 'Model']

    print(f"Best Performing Model: {best_model} (Test Accuracy: {best_test_accuracy:.4f})")

    for _, row in report_df.iterrows():
        model = row['Model']
        test_acc = row['Test_Accuracy']
        perf_gap = row['Performance_Gap']

        if model != best_model:
            diff = test_acc - best_test_accuracy
            if abs(diff) < 0.01:
                comparison = "COMPARABLE TO BEST"
            elif diff > -0.02:
                comparison = "SLIGHTLY WORSE"
            elif diff > -0.05:
                comparison = "WORSE"
            else:
                comparison = "MUCH WORSE"

            print(f"{model} vs {best_model}: {comparison} (Δ = {diff:+.4f})")

        # Performance gap analysis
        if perf_gap > 0.1:
            gap_analysis = "LARGE OVERFITTING"
        elif perf_gap > 0.05:
            gap_analysis = "MODERATE OVERFITTING"
        elif perf_gap > 0.02:
            gap_analysis = "SLIGHT OVERFITTING"
        elif perf_gap > -0.02:
            gap_analysis = "GOOD GENERALIZATION"
        else:
            gap_analysis = "UNDERFITTING"

        print(f"  {model} Generalization: {gap_analysis} (Val-Test gap: {perf_gap:+.4f})")

    # Create comprehensive visualization
    create_final_report_visualization(report_df, stats_df, validation_results, test_results, 
                                    baseline_history, xgb_history, plsr_model)

    # Save detailed reports
    report_df.to_csv(os.path.join(OUTPUT_BASE, 'final_performance_report.csv'), index=False)
    stats_df.to_csv(os.path.join(OUTPUT_BASE, 'dataset_statistics.csv'), index=False)

    # Save model performance summary
    performance_summary = {
        'best_model': best_model,
        'best_accuracy': float(best_test_accuracy),
        'total_training_samples': len(train_df),
        'total_test_samples': len(test_df),
        'report_generated': time.strftime('%Y-%m-%d %H:%M:%S'),
        'models_evaluated': models
    }

    with open(os.path.join(OUTPUT_BASE, 'performance_summary.json'), 'w') as f:
        json.dump(performance_summary, f, indent=2)

    print(f"\nFinal reports saved to: {OUTPUT_BASE}")
    print("\n" + "="*50)
    print("RICE LEAF BLAST DETECTION SYSTEM - COMPLETED SUCCESSFULLY!")
    print("="*50)

    return report_df, stats_df

def create_final_report_visualization(report_df, stats_df, validation_results, test_results, baseline_history, xgb_history, plsr_model):
    """Create comprehensive final report visualization WITH TRAINING HISTORIES"""
    # Create a larger figure to accommodate training histories
    fig = plt.figure(figsize=(20, 20))
    
    # Create grid specification with more rows for training histories
    gs = fig.add_gridspec(5, 3)  # Increased to 5 rows
    
    # Plot 1: Performance metrics comparison (test set)
    ax1 = fig.add_subplot(gs[0, 0])
    metrics = ['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1_Score', 'Test_Specificity']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']

    x = np.arange(len(report_df))
    width = 0.15

    for i, metric in enumerate(metrics):
        values = report_df[metric].values
        ax1.bar(x + (i-2)*width, values, width, label=metric_labels[i], alpha=0.8)

    ax1.set_xlabel('Models')
    ax1.set_ylabel('Score')
    ax1.set_title('Test Set Performance Metrics', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(report_df['Model'].values)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)

    # Plot 2: Validation vs Test Accuracy
    ax2 = fig.add_subplot(gs[0, 1])
    x_pos = np.arange(len(report_df))
    width = 0.35

    ax2.bar(x_pos - width/2, report_df['Val_Accuracy'], width, label='Validation', alpha=0.7, color='blue')
    ax2.bar(x_pos + width/2, report_df['Test_Accuracy'], width, label='Test', alpha=0.7, color='red')

    ax2.set_xlabel('Models')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Validation vs Test Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(report_df['Model'].values)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)

    # Plot 3: Inference Time Comparison
    ax3 = fig.add_subplot(gs[0, 2])
    inference_times = [test_results[model]['inference_time'] for model in test_results.keys()]
    bars = ax3.bar(test_results.keys(), inference_times, color='orange', alpha=0.7)
    ax3.set_xlabel('Models')
    ax3.set_ylabel('Seconds')
    ax3.set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)

    # Add value labels on bars
    for bar, time_val in zip(bars, inference_times):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{time_val:.2f}s', ha='center', va='bottom')

    # Plot 4: Dataset Distribution
    ax4 = fig.add_subplot(gs[1, 0])
    datasets = stats_df['Dataset']
    healthy_counts = stats_df['HEALTHY_Count']
    blast_counts = stats_df['LEAFBLAST_Count']

    x = np.arange(len(datasets))
    width = 0.35

    ax4.bar(x - width/2, healthy_counts, width, label='Healthy', color='green', alpha=0.7)
    ax4.bar(x + width/2, blast_counts, width, label='Leaf Blast', color='red', alpha=0.7)

    ax4.set_xlabel('Dataset')
    ax4.set_ylabel('Number of Images')
    ax4.set_title('Dataset Class Distribution', fontsize=14, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels([d.split(' ')[0] for d in datasets])
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # Plot 5: Performance Gap Analysis
    ax5 = fig.add_subplot(gs[1, 1])
    colors = ['red' if gap > 0.05 else 'orange' if gap > 0.02 else 'green' for gap in report_df['Performance_Gap']]
    bars = ax5.bar(report_df['Model'], report_df['Performance_Gap'], color=colors, alpha=0.7)
    ax5.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax5.set_xlabel('Models')
    ax5.set_ylabel('Accuracy Difference (Val - Test)')
    ax5.set_title('Generalization Performance Gap', fontsize=14, fontweight='bold')
    ax5.grid(True, alpha=0.3)

    # Add value labels
    for bar, gap in zip(bars, report_df['Performance_Gap']):
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02),
                f'{gap:+.3f}', ha='center', va='bottom' if height >= 0 else 'top')

    # Plot 6: Detailed Metrics for Best Model
    ax6 = fig.add_subplot(gs[1, 2])
    best_model = report_df.loc[report_df['Test_Accuracy'].idxmax(), 'Model'].lower()

    if best_model in test_results:
        metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
        metrics_values = [
            test_results[best_model]['accuracy'],
            test_results[best_model]['precision'],
            test_results[best_model]['recall'],
            test_results[best_model]['f1_score'],
            test_results[best_model]['specificity']
        ]

        colors = ['blue', 'green', 'orange', 'red', 'purple']
        bars = ax6.bar(metrics_names, metrics_values, color=colors, alpha=0.7)

        ax6.set_ylabel('Score')
        ax6.set_title(f'Best Model ({best_model.upper()}) Detailed Metrics', fontsize=14, fontweight='bold')
        ax6.set_ylim(0, 1)
        ax6.grid(True, alpha=0.3)

        # Add value labels
        for bar, value in zip(bars, metrics_values):
            height = bar.get_height()
            ax6.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom')

    # =============================================================================
    # TRAINING HISTORIES SECTION (NEW - Bottom 2 rows)
    # =============================================================================
    
    # Plot 7: Baseline VGG16 Accuracy History
    ax7 = fig.add_subplot(gs[2, 0])
    if baseline_history and 'accuracy' in baseline_history.history:
        epochs = range(1, len(baseline_history.history['accuracy']) + 1)
        ax7.plot(epochs, baseline_history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
        ax7.plot(epochs, baseline_history.history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
        ax7.set_xlabel('Epochs')
        ax7.set_ylabel('Accuracy')
        ax7.set_title('VGG16 Baseline - Accuracy History', fontsize=14, fontweight='bold')
        ax7.legend()
        ax7.grid(True, alpha=0.3)
        ax7.set_ylim(0, 1)
    else:
        ax7.text(0.5, 0.5, 'Baseline Accuracy History Not Available', 
                ha='center', va='center', transform=ax7.transAxes)
        ax7.set_title('VGG16 Baseline - Accuracy History', fontsize=14, fontweight='bold')

    # Plot 8: Baseline VGG16 Loss History
    ax8 = fig.add_subplot(gs[2, 1])
    if baseline_history and 'loss' in baseline_history.history:
        epochs = range(1, len(baseline_history.history['loss']) + 1)
        ax8.plot(epochs, baseline_history.history['loss'], 'b-', label='Training Loss', linewidth=2)
        ax8.plot(epochs, baseline_history.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
        ax8.set_xlabel('Epochs')
        ax8.set_ylabel('Loss')
        ax8.set_title('VGG16 Baseline - Loss History', fontsize=14, fontweight='bold')
        ax8.legend()
        ax8.grid(True, alpha=0.3)
    else:
        ax8.text(0.5, 0.5, 'Baseline Loss History Not Available', 
                ha='center', va='center', transform=ax8.transAxes)
        ax8.set_title('VGG16 Baseline - Loss History', fontsize=14, fontweight='bold')

    # Plot 9: XGBoost Training History
    ax9 = fig.add_subplot(gs[2, 2])
    if xgb_history and 'validation_0' in xgb_history and 'validation_1' in xgb_history:
        train_logloss = xgb_history['validation_0']['logloss']
        val_logloss = xgb_history['validation_1']['logloss']
        iterations = range(1, len(train_logloss) + 1)
        
        ax9.plot(iterations, train_logloss, 'g-', label='Training Log Loss', linewidth=2)
        ax9.plot(iterations, val_logloss, 'orange', label='Validation Log Loss', linewidth=2)
        ax9.set_xlabel('Boosting Rounds')
        ax9.set_ylabel('Log Loss')
        ax9.set_title('XGBoost - Training History', fontsize=14, fontweight='bold')
        ax9.legend()
        ax9.grid(True, alpha=0.3)
        
        # Add final values annotation
        final_train_loss = train_logloss[-1]
        final_val_loss = val_logloss[-1]
        ax9.annotate(f'Final Train: {final_train_loss:.3f}\nFinal Val: {final_val_loss:.3f}', 
                   xy=(0.98, 0.98), xycoords='axes fraction',
                   ha='right', va='top', fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    else:
        ax9.text(0.5, 0.5, 'XGBoost History Not Available', 
                ha='center', va='center', transform=ax9.transAxes)
        ax9.set_title('XGBoost - Training History', fontsize=14, fontweight='bold')

    # Plot 10: PLSR Component Analysis
    ax10 = fig.add_subplot(gs[3, :])  # Span entire bottom row
    if plsr_model is not None:
        try:
            # Get explained variance ratio for components
            n_components = min(20, plsr_model.x_scores_.shape[1])
            components = range(1, n_components + 1)
            
            # Calculate cumulative explained variance
            x_variance = np.var(plsr_model.x_scores_, axis=0)
            y_variance = np.var(plsr_model.y_scores_, axis=0)
            
            # Normalize to percentage
            x_variance_ratio = x_variance / np.sum(x_variance) * 100
            y_variance_ratio = y_variance / np.sum(y_variance) * 100
            
            # Plot individual variance
            width = 0.35
            x_pos = np.arange(len(components))
            
            bars1 = ax10.bar(x_pos - width/2, x_variance_ratio[:n_components], 
                           width, label='X-Variance (Features)', alpha=0.7, color='blue')
            bars2 = ax10.bar(x_pos + width/2, y_variance_ratio[:n_components], 
                           width, label='Y-Variance (Labels)', alpha=0.7, color='red')
            
            ax10.set_xlabel('PLS Components')
            ax10.set_ylabel('Explained Variance (%)')
            ax10.set_title('PLSR - Component Variance Analysis', fontsize=14, fontweight='bold')
            ax10.set_xticks(x_pos)
            ax10.set_xticklabels(components)
            ax10.legend()
            ax10.grid(True, alpha=0.3)
            
            # Add total variance explained annotation
            total_x_var = np.sum(x_variance_ratio[:n_components])
            total_y_var = np.sum(y_variance_ratio[:n_components])
            ax10.annotate(f'Total X-Variance: {total_x_var:.1f}%\nTotal Y-Variance: {total_y_var:.1f}%', 
                         xy=(0.98, 0.98), xycoords='axes fraction',
                         ha='right', va='top', fontsize=10,
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                           
        except Exception as e:
            ax10.text(0.5, 0.5, f'PLSR Analysis Error: {str(e)}', 
                     ha='center', va='center', transform=ax10.transAxes)
            ax10.set_title('PLSR - Component Analysis', fontsize=14, fontweight='bold')
    else:
        ax10.text(0.5, 0.5, 'PLSR Model Not Available', 
                 ha='center', va='center', transform=ax10.transAxes)
        ax10.set_title('PLSR - Component Analysis', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'final_comprehensive_report_unified.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# Generate final comprehensive report
print("Generating comprehensive performance report...")
final_report_df, final_stats_df = generate_comprehensive_report(
    validation_results, test_results, train_df, val_df, test_df
)

## 6. Error Analysis

In [ ]:
#################
# 2025 Deepseek #
#################
def comprehensive_error_analysis(baseline_model, plsr_model, xgb_model, X_val, y_val, 
                                val_df, X_test, y_test, test_df, cached_features):
    """Perform comprehensive error analysis across all models"""
    print("="*60)
    print("COMPREHENSIVE ERROR ANALYSIS")
    print("="*60)
    
    results = {}
    models = {
        'baseline': baseline_model,
        'plsr': plsr_model,
        'xgboost': xgb_model if XGB_AVAILABLE and xgb_model else None
    }
    
    # Analyze each model
    for model_name, model in models.items():
        if model is None:
            continue
            
        print(f"\n{'='*40}")
        print(f"ERROR ANALYSIS: {model_name.upper()}")
        print(f"{'='*40}")
        
        # Get predictions
        if model_name == 'baseline':
            y_pred_proba = model.predict(X_val, verbose=0).flatten()
            y_pred = (y_pred_proba > 0.5).astype(int)
        elif model_name == 'plsr':
            val_features = cached_features['val_scaled']
            y_pred_proba = model.predict(val_features).flatten()
            y_pred = (y_pred_proba > 0.5).astype(int)
        elif model_name == 'xgboost':
            val_features = cached_features['val_scaled']
            y_pred_proba = model.predict_proba(val_features)[:, 1]
            y_pred = model.predict(val_features)
        
        # Calculate confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        
        # Get misclassified samples
        misclassified_indices = np.where(y_pred != y_val)[0]
        
        # Analyze false positives and false negatives separately
        fp_indices = np.where((y_pred == 1) & (y_val == 0))[0]
        fn_indices = np.where((y_pred == 0) & (y_val == 1))[0]
        
        results[model_name] = {
            'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
            'misclassified_count': len(misclassified_indices),
            'fp_count': len(fp_indices),
            'fn_count': len(fn_indices),
            'fp_indices': fp_indices,
            'fn_indices': fn_indices,
            'misclassified_indices': misclassified_indices,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
        
        # Calculate error metrics
        error_rate = len(misclassified_indices) / len(y_val)
        fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
        fn_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
        
        print(f"Error Rate: {error_rate:.4f} ({len(misclassified_indices)}/{len(y_val)})")
        print(f"False Positive Rate: {fp_rate:.4f} ({fp}/{fp+tn})")
        print(f"False Negative Rate: {fn_rate:.4f} ({fn}/{fn+tp})")
        print(f"Precision: {precision_score(y_val, y_pred, zero_division=0):.4f}")
        print(f"Recall: {recall_score(y_val, y_pred, zero_division=0):.4f}")
        
        # Analyze confidence of errors
        if len(misclassified_indices) > 0:
            error_confidences = y_pred_proba[misclassified_indices]
            mean_error_confidence = np.mean(error_confidences)
            std_error_confidence = np.std(error_confidences)
            print(f"Mean confidence on errors: {mean_error_confidence:.4f} (±{std_error_confidence:.4f})")
            
            # High confidence errors (confidence > 0.8)
            high_conf_errors = error_confidences[(error_confidences > 0.8) | (error_confidences < 0.2)]
            print(f"High confidence errors (>0.8 or <0.2): {len(high_conf_errors)}/{len(error_confidences)}")
    
    # Comparative analysis across models
    print(f"\n{'='*50}")
    print("COMPARATIVE ERROR ANALYSIS ACROSS MODELS")
    print(f"{'='*50}")
    
    # Create comparison DataFrame
    comparison_data = []
    for model_name in results.keys():
        result = results[model_name]
        cm = result['confusion_matrix']
        
        # Calculate additional metrics
        precision = cm['tp'] / (cm['tp'] + cm['fp']) if (cm['tp'] + cm['fp']) > 0 else 0
        recall = cm['tp'] / (cm['tp'] + cm['fn']) if (cm['tp'] + cm['fn']) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        comparison_data.append({
            'Model': model_name.upper(),
            'Error Rate': result['misclassified_count'] / len(y_val),
            'FP Rate': cm['fp'] / (cm['fp'] + cm['tn']) if (cm['fp'] + cm['tn']) > 0 else 0,
            'FN Rate': cm['fn'] / (cm['fn'] + cm['tp']) if (cm['fn'] + cm['tp']) > 0 else 0,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'FP Count': cm['fp'],
            'FN Count': cm['fn'],
            'Total Errors': result['misclassified_count']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\nError Metrics Comparison:")
    print(comparison_df.to_string(index=False))
    
    # Visualize error analysis
    visualize_error_analysis(results, comparison_df, X_val, y_val, val_df)
    
    # Analyze model agreement on errors
    analyze_model_agreement(results, y_val)
    
    return results, comparison_df

def visualize_error_analysis(results, comparison_df, X_val, y_val, val_df):
    """Create comprehensive error analysis visualizations"""
    
    # 1. Error rate comparison
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Error rate comparison
    axes[0, 0].bar(comparison_df['Model'], comparison_df['Error Rate'], 
                   color=['blue', 'orange', 'green'])
    axes[0, 0].set_title('Error Rate Comparison', fontweight='bold')
    axes[0, 0].set_ylabel('Error Rate')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: FP vs FN rates
    x = np.arange(len(comparison_df))
    width = 0.35
    axes[0, 1].bar(x - width/2, comparison_df['FP Rate'], width, label='FP Rate', alpha=0.7)
    axes[0, 1].bar(x + width/2, comparison_df['FN Rate'], width, label='FN Rate', alpha=0.7)
    axes[0, 1].set_xlabel('Models')
    axes[0, 1].set_ylabel('Rate')
    axes[0, 1].set_title('False Positive vs False Negative Rates', fontweight='bold')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(comparison_df['Model'])
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Precision-Recall tradeoff
    for i, row in comparison_df.iterrows():
        axes[0, 2].scatter(row['Recall'], row['Precision'], s=200, label=row['Model'])
    axes[0, 2].set_xlabel('Recall')
    axes[0, 2].set_ylabel('Precision')
    axes[0, 2].set_title('Precision-Recall Tradeoff', fontweight='bold')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_xlim(0, 1)
    axes[0, 2].set_ylim(0, 1)
    
    # Plot 4: Confusion matrices
    for idx, (model_name, result) in enumerate(list(results.items())[:3]):
        if idx < 3:
            cm = np.array([[result['confusion_matrix']['tn'], result['confusion_matrix']['fp']],
                          [result['confusion_matrix']['fn'], result['confusion_matrix']['tp']]])
            
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, idx],
                       xticklabels=['Healthy', 'Blast'],
                       yticklabels=['Healthy', 'Blast'])
            axes[1, idx].set_title(f'{model_name.upper()} - Confusion Matrix', fontweight='bold')
            axes[1, idx].set_xlabel('Predicted')
            axes[1, idx].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comprehensive_error_analysis.png'),
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. Visualize specific error cases with Grad-CAM
    if 'baseline' in results and len(results['baseline']['fp_indices']) > 0:
        visualize_error_cases_with_gradcam(
            baseline_model, X_val, y_val, val_df,
            results['baseline']['fp_indices'][:4],
            results['baseline']['fn_indices'][:4],
            title="Error Case Analysis (Baseline VGG16)"
        )

def visualize_error_cases_with_gradcam(model, X_val, y_val, val_df, fp_indices, fn_indices, title):
    """Visualize error cases with Grad-CAM explanations"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}
    
    # False Positives (Predicted Blast, Actual Healthy)
    for i, idx in enumerate(fp_indices[:4]):
        img_array = X_val[idx:idx+1]
        actual_label = y_val[idx]
        pred_proba = model.predict(img_array, verbose=0)[0][0]
        
        # Generate Grad-CAM
        heatmap = enhanced_gradcam_heatmap(img_array, model)
        heatmap_resized = cv2.resize(heatmap, (224, 224))
        
        # Create overlay
        original_img = (img_array[0] * 255).astype(np.uint8)
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        superimposed = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)
        
        axes[0, i].imshow(superimposed)
        axes[0, i].set_title(f'FP: Pred={pred_proba:.2f}, Actual={class_names[actual_label]}',
                           color='red', fontweight='bold')
        axes[0, i].axis('off')
    
    # False Negatives (Predicted Healthy, Actual Blast)
    for i, idx in enumerate(fn_indices[:4]):
        img_array = X_val[idx:idx+1]
        actual_label = y_val[idx]
        pred_proba = model.predict(img_array, verbose=0)[0][0]
        
        # Generate Grad-CAM
        heatmap = enhanced_gradcam_heatmap(img_array, model)
        heatmap_resized = cv2.resize(heatmap, (224, 224))
        
        # Create overlay
        original_img = (img_array[0] * 255).astype(np.uint8)
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        superimposed = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)
        
        axes[1, i].imshow(superimposed)
        axes[1, i].set_title(f'FN: Pred={pred_proba:.2f}, Actual={class_names[actual_label]}',
                           color='red', fontweight='bold')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'error_cases_gradcam.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

def analyze_model_agreement(results, y_val):
    """Analyze where models agree/disagree on errors"""
    print(f"\n{'='*50}")
    print("MODEL AGREEMENT ANALYSIS ON ERRORS")
    print(f"{'='*50}")
    
    # Get error masks for each model
    error_masks = {}
    for model_name, result in results.items():
        error_masks[model_name] = (result['y_pred'] != y_val)
    
    # Calculate agreement
    models_list = list(error_masks.keys())
    
    if len(models_list) >= 2:
        # Pairwise agreement
        print("\nPairwise Error Agreement (% of errors where both models are wrong):")
        for i in range(len(models_list)):
            for j in range(i+1, len(models_list)):
                m1, m2 = models_list[i], models_list[j]
                agreement = np.sum(error_masks[m1] & error_masks[m2])
                total_errors_m1 = np.sum(error_masks[m1])
                total_errors_m2 = np.sum(error_masks[m2])
                
                if total_errors_m1 > 0 and total_errors_m2 > 0:
                    pct_m1 = agreement / total_errors_m1 * 100
                    pct_m2 = agreement / total_errors_m2 * 100
                    print(f"{m1.upper()} vs {m2.upper()}: {agreement} common errors")
                    print(f"  - {pct_m1:.1f}% of {m1}'s errors are also errors in {m2}")
                    print(f"  - {pct_m2:.1f}% of {m2}'s errors are also errors in {m1}")
        
        # Consensus errors (all models wrong)
        if len(models_list) == 3:
            consensus_errors = error_masks[models_list[0]] & error_masks[models_list[1]] & error_masks[models_list[2]]
            consensus_count = np.sum(consensus_errors)
            print(f"\nConsensus Errors (ALL models wrong): {consensus_count}/{len(y_val)} ({consensus_count/len(y_val)*100:.1f}%)")
            
            if consensus_count > 0:
                print("These are the most challenging cases - all models fail on them!")
                print("Indices of consensus errors:", np.where(consensus_errors)[0][:10])

# Run comprehensive error analysis
print("Running comprehensive error analysis...")
error_results, error_comparison = comprehensive_error_analysis(
    baseline_model, plsr_model, xgb_model, X_val, y_val, val_df,
    X_test, y_test, test_df, cached_features
)

In [ ]:
def analyze_hybrid_failure(error_results, X_val, y_val, val_df, baseline_model, plsr_model, xgb_model):
    """Analyze why hybrid models (PLSR/XGBoost) failed compared to baseline"""
    print("="*60)
    print("HYBRID MODEL FAILURE ANALYSIS")
    print("="*60)
    
    # Metrics to measure failure magnitude
    baseline_error = error_results['baseline']['misclassified_count']
    baseline_fp = error_results['baseline']['confusion_matrix']['fp']
    baseline_fn = error_results['baseline']['confusion_matrix']['fn']
    
    # Compare with PLSR
    if 'plsr' in error_results:
        plsr_error = error_results['plsr']['misclassified_count']
        plsr_fp = error_results['plsr']['confusion_matrix']['fp']
        plsr_fn = error_results['plsr']['confusion_matrix']['fn']
        
        error_diff = plsr_error - baseline_error
        fp_diff = plsr_fp - baseline_fp
        fn_diff = plsr_fn - baseline_fn
        
        print(f"\nPLSR vs Baseline Performance Gap:")
        print(f"Additional errors made by PLSR: {error_diff}")
        print(f"Additional FP errors: {fp_diff}")
        print(f"Additional FN errors: {fn_diff}")
        
        # Calculate error magnitude metric
        error_magnitude = error_diff / baseline_error if baseline_error > 0 else 0
        print(f"Error Magnitude (PLSR/Baseline): {error_magnitude:.2f}x more errors")
        
        # Analyze feature representation issues
        analyze_feature_representation_issues(X_val, baseline_model, plsr_model, 
                                             error_results['baseline'], error_results['plsr'])
    
    # Compare with XGBoost
    if 'xgboost' in error_results and error_results['xgboost'] is not None:
        xgb_error = error_results['xgboost']['misclassified_count']
        xgb_fp = error_results['xgboost']['confusion_matrix']['fp']
        xgb_fn = error_results['xgboost']['confusion_matrix']['fn']
        
        error_diff = xgb_error - baseline_error
        fp_diff = xgb_fp - baseline_fp
        fn_diff = xgb_fn - baseline_fn
        
        print(f"\nXGBoost vs Baseline Performance Gap:")
        print(f"Additional errors made by XGBoost: {error_diff}")
        print(f"Additional FP errors: {fp_diff}")
        print(f"Additional FN errors: {fn_diff}")
        
        # Calculate error magnitude metric
        error_magnitude = error_diff / baseline_error if baseline_error > 0 else 0
        print(f"Error Magnitude (XGBoost/Baseline): {error_magnitude:.2f}x more errors")
        
        # Analyze feature importance issues
        analyze_xgboost_feature_importance(xgb_model, error_results['xgboost'])

def analyze_feature_representation_issues(X_val, baseline_model, plsr_model, baseline_results, plsr_results):
    """Analyze what features PLSR fails to learn"""
    print(f"\n{'='*40}")
    print("FEATURE REPRESENTATION ANALYSIS (PLSR)")
    print(f"{'='*40}")
    
    # Get cases where baseline is correct but PLSR is wrong
    baseline_correct = (baseline_results['y_pred'] == y_val)
    plsr_wrong = (plsr_results['y_pred'] != y_val)
    plsr_failure_cases = np.where(baseline_correct & plsr_wrong)[0]
    
    if len(plsr_failure_cases) > 0:
        print(f"Cases where Baseline is correct but PLSR is wrong: {len(plsr_failure_cases)}")
        
        # Analyze a few cases with Grad-CAM
        sample_cases = plsr_failure_cases[:3]
        for idx in sample_cases:
            img_array = X_val[idx:idx+1]
            actual_label = y_val[idx]
            baseline_pred = baseline_results['y_pred'][idx]
            plsr_pred = plsr_results['y_pred'][idx]
            
            # Generate Grad-CAM for baseline
            baseline_heatmap = enhanced_gradcam_heatmap(img_array, baseline_model)
            
            print(f"\nCase {idx}: Actual={actual_label}, Baseline={baseline_pred}, PLSR={plsr_pred}")
            print(f"  Baseline confidence: {baseline_results['y_pred_proba'][idx]:.3f}")
            print(f"  PLSR confidence: {plsr_results['y_pred_proba'][idx]:.3f}")
            
            # Analyze heatmap characteristics
            heatmap_mean = np.mean(baseline_heatmap)
            heatmap_std = np.std(baseline_heatmap)
            print(f"  Baseline attention: mean={heatmap_mean:.3f}, std={heatmap_std:.3f}")
            
            # Identify if attention is diffuse or focused
            if heatmap_std < 0.1:
                print("  Observation: Baseline attention is DIFFUSE (spread across image)")
                print("  Hypothesis: PLSR may struggle with diffuse patterns, preferring localized features")
            else:
                print("  Observation: Baseline attention is FOCUSED (specific regions)")
                print("  Hypothesis: PLSR may miss subtle localized patterns that VGG16 captures")
    
    # Analyze PLSR coefficients to see what features it prioritizes
    if hasattr(plsr_model, 'coef_'):
        coef_magnitude = np.abs(plsr_model.coef_.flatten())
        top_features_idx = np.argsort(coef_magnitude)[-10:]  # Top 10 features
        print(f"\nPLSR Top 10 Feature Importance Indices: {top_features_idx}")
        print(f"PLSR Coefficient Stats: mean={np.mean(coef_magnitude):.3f}, std={np.std(coef_magnitude):.3f}")
        
        # If coefficients are very uniform, PLSR isn't focusing on specific features
        coefficient_entropy = -np.sum(coef_magnitude * np.log(coef_magnitude + 1e-10))
        if coefficient_entropy > np.log(len(coef_magnitude)) * 0.8:
            print("  Issue: PLSR coefficients have high entropy - not focusing on specific features")
            print("  This suggests PLSR is not effectively learning discriminative features")

def analyze_xgboost_feature_importance(xgb_model, xgb_results):
    """Analyze XGBoost feature importance patterns"""
    print(f"\n{'='*40}")
    print("XGBOOST FEATURE IMPORTANCE ANALYSIS")
    print(f"{'='*40}")
    
    if hasattr(xgb_model, 'feature_importances_'):
        importances = xgb_model.feature_importances_
        
        # Analyze importance distribution
        top_n = 20
        sorted_idx = np.argsort(importances)[::-1]
        
        print(f"Top {top_n} Feature Importances:")
        for i in range(min(top_n, len(importances))):
            print(f"  Feature {sorted_idx[i]}: {importances[sorted_idx[i]]:.4f}")
        
        # Calculate importance concentration
        gini_coefficient = calculate_gini_coefficient(importances)
        print(f"\nFeature Importance Concentration (Gini Coefficient): {gini_coefficient:.3f}")
        
        if gini_coefficient < 0.3:
            print("  Issue: Feature importance is too uniform")
            print("  XGBoost is not focusing on specific discriminative features")
            print("  This suggests the VGG16 features may not be optimally structured for tree-based methods")
        
        # Check if important features are from specific channels
        if len(importances) == 25088:  # 7x7x512
            # Reshape to spatial dimensions
            spatial_importance = importances.reshape(7, 7, 512)
            channel_importance = spatial_importance.mean(axis=(0, 1))
            
            top_channels = np.argsort(channel_importance)[-5:]
            print(f"\nTop 5 Important Channels: {top_channels}")
            print(f"Mean channel importance: {np.mean(channel_importance):.4f}")
            
            if np.mean(channel_importance) < 0.002:
                print("  Issue: Channel importance is very low across the board")
                print("  XGBoost may be treating all features equally, indicating poor feature discrimination")

def calculate_gini_coefficient(values):
    """Calculate Gini coefficient to measure inequality/concentration"""
    values = np.sort(values)
    n = len(values)
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * values)) / (n * np.sum(values))

# Run hybrid failure analysis
print("Analyzing hybrid model failures...")
analyze_hybrid_failure(error_results, X_val, y_val, val_df, baseline_model, plsr_model, xgb_model)

## 7. Severity by Bucketing

In [ ]:
#################
# 2025 Deepseek #
#################
def implement_severity_metric(y_pred_proba, severity_buckets=None):
    """
    Convert binary probabilities to multi-class severity levels
    
    Parameters:
    -----------
    y_pred_proba : array, predicted probabilities for leaf blast
    severity_buckets : dict, threshold definitions for severity levels
    
    Returns:
    --------
    severity_classes : array, severity labels (0=Healthy, 1=Mild, 2=Moderate, 3=Severe)
    """
    if severity_buckets is None:
        # Panelist recommendation: bucket probabilities
        severity_buckets = {
            'Healthy': (0.0, 0.25),      # 0-25% probability
            'Mild': (0.25, 0.50),        # 25-50% probability
            'Moderate': (0.50, 0.75),    # 50-75% probability
            'Severe': (0.75, 1.0)        # 75-100% probability
        }
    
    severity_classes = np.zeros_like(y_pred_proba, dtype=int)
    
    # Map probabilities to severity classes
    severity_classes = np.zeros_like(y_pred_proba, dtype=int)
    severity_classes[(y_pred_proba >= 0.25) & (y_pred_proba < 0.50)] = 1  # Mild
    severity_classes[(y_pred_proba >= 0.50) & (y_pred_proba < 0.75)] = 2  # Moderate
    severity_classes[y_pred_proba >= 0.75] = 3  # Severe
    
    return severity_classes, severity_buckets

def evaluate_severity_metrics(models_dict, X_test, y_test, test_df, cached_features):
    """
    Evaluate models using severity metrics instead of binary classification
    
    Parameters:
    -----------
    models_dict : dict, {'baseline': model, 'plsr': model, 'xgboost': model}
    X_test : array, test images
    y_test : array, binary test labels
    test_df : DataFrame, test metadata
    cached_features : dict, cached VGG16 features
    """
    print("="*60)
    print("SEVERITY METRIC EVALUATION")
    print("="*60)
    
    severity_results = {}
    
    for model_name, model in models_dict.items():
        if model is None:
            continue
            
        print(f"\n{model_name.upper()} - Severity Analysis:")
        
        # Get predicted probabilities
        if model_name == 'baseline':
            y_pred_proba = model.predict(X_test, verbose=0).flatten()
        elif model_name == 'plsr':
            test_features = cached_features['test_scaled']
            y_pred_proba = model.predict(test_features).flatten()
            # PLSR outputs can be outside [0,1], so normalize
            y_pred_proba = (y_pred_proba - y_pred_proba.min()) / (y_pred_proba.max() - y_pred_proba.min())
        elif model_name == 'xgboost':
            test_features = cached_features['test_scaled']
            y_pred_proba = model.predict_proba(test_features)[:, 1]
        
        # Convert to severity classes
        severity_classes, buckets = implement_severity_metric(y_pred_proba)
        
        # Analyze distribution
        unique_classes, class_counts = np.unique(severity_classes, return_counts=True)
        class_names = ['Healthy', 'Mild', 'Moderate', 'Severe']
        
        print(f"Severity Distribution:")
        for cls, count in zip(unique_classes, class_counts):
            pct = count / len(severity_classes) * 100
            print(f"  {class_names[cls]}: {count} ({pct:.1f}%)")
        
        # For leaf blast samples (y_test == 1), analyze severity predictions
        blast_indices = np.where(y_test == 1)[0]
        if len(blast_indices) > 0:
            blast_severity = severity_classes[blast_indices]
            print(f"\nFor actual Leaf Blast samples (n={len(blast_indices)}):")
            
            for severity_level in [1, 2, 3]:  # Mild, Moderate, Severe
                count = np.sum(blast_severity == severity_level)
                pct = count / len(blast_indices) * 100
                print(f"  Predicted as {class_names[severity_level]}: {count} ({pct:.1f}%)")
            
            # Calculate average severity for actual blast cases
            avg_severity = np.mean(blast_severity)
            print(f"  Average predicted severity: {avg_severity:.2f}")
            
            # Calculate severity consistency metric
            severity_std = np.std(blast_severity)
            print(f"  Severity consistency (std): {severity_std:.2f}")
        
        # For healthy samples, check false severity predictions
        healthy_indices = np.where(y_test == 0)[0]
        if len(healthy_indices) > 0:
            healthy_severity = severity_classes[healthy_indices]
            false_severity = np.sum(healthy_severity > 0)  # Any severity > 0 (not healthy)
            false_severity_pct = false_severity / len(healthy_indices) * 100
            print(f"\nHealthy samples incorrectly given severity rating: {false_severity}/{len(healthy_indices)} ({false_severity_pct:.1f}%)")
        
        # Store results
        severity_results[model_name] = {
            'severity_classes': severity_classes,
            'severity_buckets': buckets,
            'y_pred_proba': y_pred_proba,
            'distribution': dict(zip(unique_classes, class_counts))
        }
    
    # Visualize severity distributions across models
    visualize_severity_distributions(severity_results, y_test)
    
    return severity_results

def visualize_severity_distributions(severity_results, y_test):
    """Visualize severity distributions across models"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    models = list(severity_results.keys())
    class_names = ['Healthy', 'Mild', 'Moderate', 'Severe']
    
    # Plot 1: Overall severity distribution
    for i, model_name in enumerate(models):
        dist = severity_results[model_name]['distribution']
        values = [dist.get(j, 0) for j in range(4)]
        pct_values = [v / sum(values) * 100 for v in values]
        
        axes[0, 0].bar(np.arange(4) + i*0.2, pct_values, width=0.2, 
                      label=model_name.upper(), alpha=0.7)
    
    axes[0, 0].set_xlabel('Severity Class')
    axes[0, 0].set_ylabel('Percentage (%)')
    axes[0, 0].set_title('Overall Severity Distribution by Model', fontweight='bold')
    axes[0, 0].set_xticks(np.arange(4) + 0.3)
    axes[0, 0].set_xticklabels(class_names)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Severity for actual Leaf Blast cases
    blast_indices = np.where(y_test == 1)[0]
    if len(blast_indices) > 0:
        for i, model_name in enumerate(models):
            severity_classes = severity_results[model_name]['severity_classes']
            blast_severity = severity_classes[blast_indices]
            
            values = [np.sum(blast_severity == j) for j in range(1, 4)]  # Only Mild, Moderate, Severe
            pct_values = [v / len(blast_indices) * 100 for v in values]
            
            axes[0, 1].bar(np.arange(3) + i*0.2, pct_values, width=0.2,
                          label=model_name.upper(), alpha=0.7)
        
        axes[0, 1].set_xlabel('Severity Class')
        axes[0, 1].set_ylabel('Percentage (%)')
        axes[0, 1].set_title('Severity Distribution for Actual Leaf Blast', fontweight='bold')
        axes[0, 1].set_xticks(np.arange(3) + 0.3)
        axes[0, 1].set_xticklabels(['Mild', 'Moderate', 'Severe'])
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Probability distributions
    for i, model_name in enumerate(models):
        y_pred_proba = severity_results[model_name]['y_pred_proba']
        
        # Separate probabilities for healthy and blast cases
        healthy_proba = y_pred_proba[y_test == 0]
        blast_proba = y_pred_proba[y_test == 1]
        
        axes[1, i].hist(healthy_proba, bins=20, alpha=0.5, label='Healthy', color='green')
        axes[1, i].hist(blast_proba, bins=20, alpha=0.5, label='Leaf Blast', color='red')
        axes[1, i].set_xlabel('Predicted Probability')
        axes[1, i].set_ylabel('Count')
        axes[1, i].set_title(f'{model_name.upper()} - Probability Distribution', fontweight='bold')
        axes[1, i].legend()
        axes[1, i].grid(True, alpha=0.3)
        
        # Add severity thresholds
        axes[1, i].axvline(x=0.25, color='orange', linestyle='--', alpha=0.5)
        axes[1, i].axvline(x=0.50, color='orange', linestyle='--', alpha=0.5)
        axes[1, i].axvline(x=0.75, color='orange', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'severity_metric_analysis.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# Run severity metric evaluation
print("Implementing severity metrics...")
models_dict = {
    'baseline': baseline_model,
    'plsr': plsr_model,
    'xgboost': xgb_model if XGB_AVAILABLE and xgb_model else None
}

severity_results = evaluate_severity_metrics(
    models_dict, X_test, y_test, test_df, cached_features
)

## 8 Super Final Report

In [ ]:
def generate_conclusions_recommendations(error_results, severity_results, error_comparison):
    """Generate final conclusions and recommendations based on analysis"""
    print("="*60)
    print("FINAL CONCLUSIONS & RECOMMENDATIONS")
    print("="*60)
    
    conclusions = []
    recommendations = []
    
    # 1. Error Analysis Conclusions
    best_model = error_comparison.loc[error_comparison['Error Rate'].idxmin(), 'Model']
    worst_model = error_comparison.loc[error_comparison['Error Rate'].idxmax(), 'Model']
    
    conclusions.append("1. MODEL PERFORMANCE HIERARCHY:")
    conclusions.append(f"   - Best Model: {best_model} (Lowest Error Rate)")
    conclusions.append(f"   - Worst Model: {worst_model} (Highest Error Rate)")
    
    # 2. Hybrid Model Failure Analysis
    if 'plsr' in error_results:
        plsr_error = error_comparison.loc[error_comparison['Model'] == 'PLSR', 'Error Rate'].values[0]
        baseline_error = error_comparison.loc[error_comparison['Model'] == 'BASELINE', 'Error Rate'].values[0]
        plsr_failure_rate = (plsr_error - baseline_error) / baseline_error * 100
        
        conclusions.append("\n2. HYBRID MODEL FAILURE ANALYSIS:")
        conclusions.append(f"   - PLSR underperforms baseline by {plsr_failure_rate:.1f}%")
        conclusions.append("   - Primary failure modes:")
        conclusions.append("     * Feature dimensionality reduction loses discriminative information")
        conclusions.append("     * Linear PLSR cannot capture complex non-linear relationships")
        conclusions.append("     * Grad-CAM shows PLSR fails on diffuse attention patterns")
    
    # 3. Severity Metric Insights
    if severity_results:
        conclusions.append("\n3. SEVERITY METRIC FINDINGS:")
        
        for model_name, result in severity_results.items():
            dist = result['distribution']
            severity_classes = result['severity_classes']
            
            # Calculate severity spread
            non_zero_severity = np.sum(severity_classes > 0)
            severity_spread = non_zero_severity / len(severity_classes) * 100
            
            conclusions.append(f"   - {model_name.upper()}: {severity_spread:.1f}% of samples assigned non-zero severity")
            
            # Check if model can distinguish severity levels
            if 1 in dist and 3 in dist:  # Has both mild and severe predictions
                mild_pct = dist[1] / sum(dist.values()) * 100
                severe_pct = dist[3] / sum(dist.values()) * 100
                conclusions.append(f"     * Severity range: Mild ({mild_pct:.1f}%) to Severe ({severe_pct:.1f}%)")
    
    # 4. Recommendations
    recommendations.append("\nRECOMMENDATIONS FOR FUTURE WORK:")
    recommendations.append("1. Data Collection:")
    recommendations.append("   - Collect more diverse leaf blast severity examples")
    recommendations.append("   - Include images with varying infection stages")
    
    recommendations.append("\n2. Model Architecture:")
    recommendations.append("   - Try alternative feature extractors (ResNet50, EfficientNet)")
    recommendations.append("   - Experiment with attention mechanisms for better localization")
    recommendations.append("   - Use ensemble methods combining VGG16 with other architectures")
    
    recommendations.append("\n3. Feature Engineering:")
    recommendations.append("   - Extract hand-crafted features (texture, color, shape)")
    recommendations.append("   - Combine CNN features with traditional image features")
    
    recommendations.append("\n4. Severity Quantification:")
    recommendations.append("   - Implement regression-based severity scoring")
    recommendations.append("   - Use multi-task learning for detection + severity estimation")
    recommendations.append("   - Collect ground truth severity labels for training")
    
    # Print conclusions and recommendations
    print("\nCONCLUSIONS:")
    for conclusion in conclusions:
        print(conclusion)
    
    print("\n" + "="*50)
    print("RECOMMENDATIONS:")
    for recommendation in recommendations:
        print(recommendation)
    
    # Save to file
    with open(os.path.join(OUTPUT_BASE, 'conclusions_recommendations.txt'), 'w') as f:
        f.write("="*60 + "\n")
        f.write("FINAL CONCLUSIONS & RECOMMENDATIONS\n")
        f.write("="*60 + "\n\n")
        
        f.write("CONCLUSIONS:\n")
        for conclusion in conclusions:
            f.write(conclusion + "\n")
        
        f.write("\n" + "="*50 + "\n")
        f.write("RECOMMENDATIONS:\n")
        for recommendation in recommendations:
            f.write(recommendation + "\n")
    
    print(f"\n✓ Conclusions and recommendations saved to: {OUTPUT_BASE}/conclusions_recommendations.txt")

# Generate final conclusions
print("Generating final conclusions and recommendations...")
generate_conclusions_recommendations(error_results, severity_results, error_comparison)